# Food Recognition - Food 101

# Imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
import os
import random
from PIL import Image

## Data

In [ ]:
import os

# ── 1. Try Google Drive mount ────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive"
    print("Google Drive mounted.")
except NotImplementedError:
    DRIVE_ROOT = "/content"
    print("WARNING: drive.mount() not supported — will download dataset if needed.")
except Exception as e:
    DRIVE_ROOT = "/content"
    print(f"WARNING: drive.mount() failed ({e}) — will download dataset if needed.")

BASE_DIR         = os.path.join(DRIVE_ROOT, "Dataset")
meta_dir         = os.path.join(BASE_DIR, "meta/meta")
images_dir       = os.path.join(BASE_DIR, "images")
class_names_path = os.path.join(meta_dir, "classes.txt")
train_list_path  = os.path.join(meta_dir, "train.txt")
test_list_path   = os.path.join(meta_dir, "test.txt")

# ── 2. Auto-download Food-101 if dataset is missing ──────────────────────────
if not os.path.isfile(class_names_path):
    print("Dataset not found — downloading Food-101 (~4.6 GB, takes ~5-10 min on Colab) ...")
    import subprocess, tarfile
    _dl = "/content/food-101.tar.gz"
    _extract = "/content/food-101-raw"
    os.makedirs(_extract, exist_ok=True)
    subprocess.run(
        ["wget", "-q", "-O", _dl,
         "http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz"],
        check=True
    )
    print("Extracting ...")
    with tarfile.open(_dl) as tar:
        tar.extractall(_extract)
    # food-101.tar.gz extracts to food-101/{images,meta}
    _food101_dir = os.path.join(_extract, "food-101")
    os.makedirs(BASE_DIR, exist_ok=True)
    os.makedirs(meta_dir, exist_ok=True)
    # Symlink images and meta into the expected layout
    if not os.path.exists(images_dir):
        os.symlink(os.path.join(_food101_dir, "images"), images_dir)
    if not os.path.exists(class_names_path):
        os.symlink(os.path.join(_food101_dir, "meta/classes.txt"), class_names_path)
    if not os.path.exists(train_list_path):
        os.symlink(os.path.join(_food101_dir, "meta/train.txt"), train_list_path)
    if not os.path.exists(test_list_path):
        os.symlink(os.path.join(_food101_dir, "meta/test.txt"), test_list_path)
    print("Food-101 ready.")

print(f"BASE_DIR : {BASE_DIR}")
print(f"Exists   : {os.path.isdir(BASE_DIR)}")


In [ ]:
expected_files = {
    "main_dir": BASE_DIR,
    "meta_dir": meta_dir,
    "images_dir": images_dir,
    "classes_file": class_names_path,
    "train_file": train_list_path,
    "test_file": test_list_path
}

print("--- Verifying BASE_DIR setup ---")
all_paths_exist = True
for name, path in expected_files.items():
    if os.path.exists(path):
        print(f"✓ {name} exists at: {path}")
    else:
        print(f"✗ {name} NOT FOUND at: {path}")
        all_paths_exist = False

if all_paths_exist:
    print("All expected data paths seem correct.")
else:
    print("Please review the `BASE_DIR` variable or your dataset's location. The `FileNotFoundError` from earlier might indicate an issue with how the dataset is mounted or its internal structure.")

**Load Class Names**

In [ ]:
import os, subprocess, tarfile

def _download_food101():
    """Download and extract Food-101 into BASE_DIR if not already present."""
    _dl  = "/content/food-101.tar.gz"
    _raw = "/content/food-101-raw"
    if not os.path.exists(_dl):
        print("Downloading Food-101 (~4.6 GB) ...")
        subprocess.run(
            ["wget", "-q", "--show-progress", "-O", _dl,
             "http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz"],
            check=True
        )
    print("Extracting Food-101 ...")
    os.makedirs(_raw, exist_ok=True)
    with tarfile.open(_dl) as tar:
        tar.extractall(_raw)
    _src = os.path.join(_raw, "food-101")
    os.makedirs(meta_dir, exist_ok=True)
    # Symlink images + meta files into expected layout
    pairs = [
        (os.path.join(_src, "images"),           images_dir),
        (os.path.join(_src, "meta/classes.txt"), class_names_path),
        (os.path.join(_src, "meta/train.txt"),   train_list_path),
        (os.path.join(_src, "meta/test.txt"),    test_list_path),
    ]
    for src, dst in pairs:
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            os.symlink(src, dst)
    print("Food-101 ready.")

# Auto-download if dataset is missing
if not os.path.isfile(class_names_path):
    print(f"'{class_names_path}' not found — auto-downloading Food-101 ...")
    _download_food101()

with open(class_names_path, 'r') as f:
    class_names = [line.strip() for line in f.readlines()]

print(f"Total classes : {len(class_names)}")
print(f"First 5 classes: {class_names[:5]}")
print(f"Last 5 classes : {class_names[-5:]}")


**Load Test/Train Data**

In [ ]:
try:
    with open(train_list_path, 'r') as f:
        train_files = [line.strip() for line in f.readlines()]

    with open(test_list_path, 'r') as f:
        test_files = [line.strip() for line in f.readlines()]

    print(f"Total training images (from train.txt): {len(train_files)}")
    print(f"Total test images (from test.txt): {len(test_files)}")
    print(f"Example training path: {train_files[0]}")
    print(f"Example test path: {test_files[0]}\n")

except FileNotFoundError:
    print(f"ERROR: 'train.txt' or 'test.txt' not found at {meta_dir}")
    raise

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for i , ax in enumerate(axes.flat):
    random_img = random.choice(train_files)
    class_name = random_img.split('/')[0]
    img_path = os.path.join(images_dir , f"{random_img}.jpg")
    try:
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(class_name.replace("_", " ").title(), fontsize=12)
        ax.axis('off')
    except FileNotFoundError:
        print(f"Warning: Could not find image {img_path}. Skipping.")
    except Exception as e:
        print(f"Error loading {img_path}: {e}")

plt.tight_layout()
plt.show()

In [ ]:
print("\n--- Visualizing Intra-Class Variation (Example: 'Sushi') ---")
target_class = "sushi"
class_images = [f for f in train_files if f.startswith(target_class)]
if class_images:
    sample_images = random.sample(class_images, min(5, len(class_images)))
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    fig.suptitle(f"Samples for: {target_class.replace('_', ' ').title()}", fontsize=16)

    for ax, img_stem in zip(axes, sample_images):
        img_path = os.path.join(images_dir, f"{img_stem}.jpg")
        try:
            img = Image.open(img_path)
            ax.imshow(img)
            ax.axis('off')
        except FileNotFoundError:
            ax.set_title("Not Found")
    plt.show()
else:
    print(f"Could not find any training images for class '{target_class}'.")

# Training

In [ ]:
pip install timm

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler # For Mixed Precision
from torch.amp import autocast # Updated import for Mixed Precision
import timm # For SOTA models, augmentations, and optimizers
from timm.data import Mixup
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from PIL import Image
from tqdm import tqdm
import numpy as np

# Configuration

In [ ]:
BASE_DIR = "/content/drive/MyDrive/Dataset"
# We use a Vision Transformer (ViT) pre-trained on ImageNet-21k for max accuracy.
# If you OOM (Out of Memory), try 'tf_efficientnetv2_m.in21k' (a valid timm tag)
MODEL_NAME = "vit_base_patch16_224.orig_in21k"
IMG_SIZE = 224 # Input size for this model
NUM_CLASSES = 101
BATCH_SIZE = 128
EPOCHS = 30
LR = 1e-4 * (BATCH_SIZE / 128)
WEIGHT_DECAY = 0.05


# --- SOTA Regularization Config ---
LABEL_SMOOTHING = 0.1
MIXUP_PROB = 1.0     # Probability of applying Mixup or CutMix
MIXUP_ALPHA = 0.8    # Mixup alpha
CUTMIX_ALPHA = 1.0   # CutMix alpha



# --- System Config ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Use all available CPU cores for data loading to prevent bottlenecks
NUM_WORKERS = os.cpu_count()
print(f"Using device: {DEVICE}")
print(f"Using {torch.cuda.device_count()} GPUs!")
print(f"Using {NUM_WORKERS} workers for data loading.")

CHECKPOINT_PATH = "/content/drive/MyDrive/food41/latest_checkpoint.pth"

meta_dir = os.path.join(BASE_DIR, "meta/meta")
images_dir = os.path.join(BASE_DIR, "images")
class_names_path = os.path.join(meta_dir, "classes.txt")
train_list_path = os.path.join(meta_dir, "train.txt")
test_list_path = os.path.join(meta_dir, "test.txt")

In [ ]:
def accuracy(output, target, topk=(1, 5)):
    """Computes the accuracy over the k top predictions for the specified values of k"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

In [ ]:
class Food101Dataset(Dataset):
    def __init__(self, file_list_path, class_list_path, transform=None):

        with open(class_list_path, 'r') as f:
            self.class_names = [line.strip() for line in f.readlines()]
        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}

        with open(file_list_path, 'r') as f:
            self.file_list = [line.strip() for line in f.readlines()]

        self.transform = transform
        print(f"Loaded {len(self.file_list)} image paths.")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):

        img_stem = self.file_list[idx]

        class_name = img_stem.split('/')[0]

        # Get class index
        label = self.class_to_idx[class_name]

        # Construct full path
        img_path = os.path.join(images_dir, f"{img_stem}.jpg")

        # Load image
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image {img_path}: {e}. Using a fallback.")
            # Return a dummy image and label on error
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))
            label = 0 # Return a default label

        # Apply transforms
        if self.transform:
            img = self.transform(img)

        return img, label

## Data Augmentation

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.08, 1.0)),
        transforms.RandomHorizontalFlip(),
        # Apply RandAugment
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
    ]),
    'val': transforms.Compose([
        transforms.Resize(IMG_SIZE + 32), # Resize to 256
        transforms.CenterCrop(IMG_SIZE),  # Crop to 224
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
    ]),
}

In [ ]:
# Create Datasets
train_dataset = Food101Dataset(train_list_path, class_names_path, transform=data_transforms['train'])
val_dataset = Food101Dataset(test_list_path, class_names_path, transform=data_transforms['val'])

# Create DataLoaders
# pin_memory=True speeds up CPU-to-GPU data transfer
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

## Mix-Up Config

In [ ]:
mixup_fn = Mixup(
    mixup_alpha=MIXUP_ALPHA,
    cutmix_alpha=CUTMIX_ALPHA,
    prob=MIXUP_PROB,
    label_smoothing=LABEL_SMOOTHING,
    num_classes=NUM_CLASSES
)

In [ ]:
print(f"Loading model: {MODEL_NAME}")
# Load pre-trained SOTA model
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES
)

In [ ]:
if torch.cuda.device_count() > 1:
    print(f"Activating DataParallel for {torch.cuda.device_count()} GPUs.")
    model = nn.DataParallel(model)

model.to(DEVICE)

In [ ]:
# Loss Function
# Note: We use standard CrossEntropyLoss here because
# the 'mixup_fn' handles label smoothing for us.
loss_fn = nn.CrossEntropyLoss()

# Optimizer (AdamW is standard for Transformers)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

In [ ]:
# Scheduler (OneCycleLR provides SOTA warmup and annealing)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader)
)

In [ ]:
# Mixed Precision Scaler
scaler = GradScaler()

## Training Loop

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler # For Mixed Precision
from torch.amp import autocast # Updated import for Mixed Precision
import timm # For SOTA models, augmentations, and optimizers
from timm.data import Mixup
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from PIL import Image
from tqdm import tqdm
import numpy as np

# --- 1. CONFIGURATION & HYPERPARAMETERS ---

# !!! IMPORTANT: Update this path to your dataset !!!
BASE_DIR = "/content/drive/MyDrive/Dataset"

# --- Model & Image Config ---
MODEL_NAME = "vit_base_patch16_224.orig_in21k"  # FIX: valid tag, unified with inference + Nutrisense bridge
IMG_SIZE = 224 # Input size for this model
NUM_CLASSES = 101

# --- Training Config ---
BATCH_SIZE = 4 # Reduced further due to OutOfMemoryError
EPOCHS = 30
LR = 1e-4 * (BATCH_SIZE / 128)
WEIGHT_DECAY = 0.05

# --- SOTA Regularization Config ---
LABEL_SMOOTHING = 0.1
MIXUP_PROB = 1.0
MIXUP_ALPHA = 0.8
CUTMIX_ALPHA = 1.0

# --- System Config ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# *** FIX FOR "AW SNAP" CRASH ***
NUM_WORKERS = 2 # Use a safe number, reduced from 3 based on warning
print(f"Using device: {DEVICE}")
print(f"Using {torch.cuda.device_count()} GPUs!")
print(f"Using {NUM_WORKERS} workers for data loading.")

# --- Define Checkpoint Path ---
CHECKPOINT_PATH = "latest_checkpoint.pth"

# Define paths
meta_dir = os.path.join(BASE_DIR, "meta/meta")
images_dir = os.path.join(BASE_DIR, "images")
class_names_path = os.path.join(meta_dir, "classes.txt")
train_list_path = os.path.join(meta_dir, "train.txt")
test_list_path = os.path.join(meta_dir, "test.txt")


# --- 2. HELPER FUNCTION FOR ACCURACY ---
def accuracy(output, target, topk=(1, 5)):
    """Computes the accuracy over the k top predictions for the specified values of k"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res


# --- 3. EFFICIENT DATASET CLASS ---
class Food101Dataset(Dataset):
    def __init__(self, file_list_path, class_list_path, transform=None):

        with open(class_list_path, 'r') as f:
            self.class_names = [line.strip() for line in f.readlines()]
        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}

        with open(file_list_path, 'r') as f:
            self.file_list = [line.strip() for line in f.readlines()]

        self.transform = transform
        print(f"Loaded {len(self.file_list)} image paths.")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_stem = self.file_list[idx]
        class_name = img_stem.split('/')[0]
        label = self.class_to_idx[class_name]
        img_path = os.path.join(images_dir, f"{img_stem}.jpg")

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image {img_path}: {e}. Using a fallback.")
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))
            label = 0

        if self.transform:
            img = self.transform(img)

        return img, label

# --- 4. DATA AUGMENTATION (SOTA) ---
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.08, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
    ]),
    'val': transforms.Compose([
        transforms.Resize(IMG_SIZE + 32),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
    ]),
}

print("Creating Datasets and DataLoaders...")
train_dataset = Food101Dataset(train_list_path, class_names_path, transform=data_transforms['train'])
val_dataset = Food101Dataset(test_list_path, class_names_path, transform=data_transforms['val'])

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    # *** FIX FOR "AW SNAP" CRASH ***
    num_workers=NUM_WORKERS, pin_memory=False
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    # *** FIX FOR "AW SNAP" CRASH ***
    num_workers=NUM_WORKERS, pin_memory=False
)

# --- 5. INITIALIZE MIXUP/CUTMIX ---
mixup_fn = Mixup(
    mixup_alpha=MIXUP_ALPHA,
    cutmix_alpha=CUTMIX_ALPHA,
    prob=MIXUP_PROB,
    label_smoothing=LABEL_SMOOTHING,
    num_classes=NUM_CLASSES
)

# --- 6. MODEL, LOSS, OPTIMIZER, SCHEDULER ---

print(f"Loading model: {MODEL_NAME}")
# Load model on CPU first
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES
).float() # Explicitly convert model parameters to float32

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader)
)

# *** FIX FOR FUTUREWARNING ***
scholar = GradScaler()

# --- Checkpoint Loading Logic ---
start_epoch = 0
best_acc1 = 0.0

if os.path.exists(CHECKPOINT_PATH):
    print(f"Checkpoint found! Loading from {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')

    # Load states
    model.load_state_dict(checkpoint['model_state_dict'])
    # Explicitly re-cast model to float32 after loading state_dict, just in case
    model = model.float()
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scholar.load_state_dict(checkpoint['scholar_state_dict'])

    start_epoch = checkpoint['epoch']
    best_acc1 = checkpoint['best_acc1']

    print(f"Successfully loaded. Resuming training from epoch {start_epoch + 1}")
else:
    print("No checkpoint found. Starting training from scratch.")

# Move the *base model* to the primary device first
model.to(DEVICE)
# Explicitly ensure model parameters are float32 on the device
model = model.float()

# --- Explicit fix for known timm/ViT mixed-precision issue ---
# Ensure the patch embedding projection operates in float32
# This addresses cases where model.float() might not be fully effective for all sub-parameters
if hasattr(model, 'patch_embed') and hasattr(model.patch_embed, 'proj'):
    print(f"PatchEmbed.proj module parameters dtypes BEFORE explicit cast:")
    for name, param in model.patch_embed.proj.named_parameters():
        print(f"  {name}: {param.dtype}")

    # Convert the entire proj module to float32 and move to device
    model.patch_embed.proj = model.patch_embed.proj.float().to(DEVICE)

    print(f"PatchEmbed.proj module parameters dtypes AFTER explicit cast:")
    for name, param in model.patch_embed.proj.named_parameters():
        print(f"  {name}: {param.dtype}")
# --- End explicit fix ---

# --- *** NEW FIX FOR OPTIMIZER STATE *** ---
# Manually move optimizer state to GPU after loading checkpoint
print("Moving optimizer state to GPU (if resuming)...")
for state in optimizer.state.values():
    for k, v in state.items():
        if isinstance(v, torch.Tensor):
            state[k] = v.to(DEVICE)
# --- *** END OF FIX *** ---

# Now, wrap in DataParallel (if using multi-GPU)
if torch.cuda.device_count() > 1:
    print(f"Activating DataParallel for {torch.cuda.device_count()} GPUs.")
    model = nn.DataParallel(model)

# --- 7. THE TRAINING LOOP ---
print("--- Starting Training ---")

for epoch in range(start_epoch, EPOCHS):
    start_time = time.time()

    # --- Training Phase ---
    model.train()
    train_loss = 0.0

    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)

    for images, labels in train_loop:
        # Let DataParallel handle scattering data from CPU
        images, labels_mixed = mixup_fn(images, labels)

        # Move images to the device *before* passing to the model
        images = images.to(DEVICE, non_blocking=True)

        with autocast('cuda', dtype=torch.float16):
            outputs = model(images)
            loss = loss_fn(outputs, labels_mixed.to(DEVICE, non_blocking=True))

        scholar.scale(loss).backward()
        scholar.step(optimizer)
        scholar.update()
        optimizer.zero_grad()
        scheduler.step()

        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item(), lr=optimizer.param_groups[0]['lr'])

    avg_train_loss = train_loss / len(train_loader)

    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    val_acc1 = 0.0
    val_acc5 = 0.0

    val_loop = tqdm(val_loader, desc="           [Val]", leave=False)

    with torch.no_grad():
        for images, labels in val_loop:
            labels = labels.to(DEVICE, non_blocking=True);

            # Move images to the device *before* passing to the model
            images = images.to(DEVICE, non_blocking=True)

            with autocast('cuda', dtype=torch.float16):
                outputs = model(images)
                loss = loss_fn(outputs, labels)

            val_loss += loss.item()

            acc1, acc5 = accuracy(outputs, labels, topk=(1, 15))
            val_acc1 += acc1.item()
            val_acc5 += acc5.item()

    avg_val_loss = val_loss / len(val_loader)
    avg_val_acc1 = val_acc1 / len(val_loader)
    avg_val_acc5 = val_acc5 / len(val_loader)

    epoch_time = time.time() - start_time

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Time: {epoch_time:.2f}s | "
          f"LR: {optimizer.param_groups[0]['lr']:.1e} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc@1: {avg_val_acc1:.2f}% | "
          f"Val Acc@5: {avg_val_acc5:.2f}%")

    # --- Save Full Checkpoint Every Epoch ---

    model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()

    checkpoint = {
        'epoch': epoch + 1, # Save the *next* epoch number to start from
        'model_state_dict': model_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scholar_state_dict': scholar.state_dict(),
        'best_acc1': best_acc1,
    }

    torch.save(checkpoint, CHECKPOINT_PATH)

    # --- Save the best model (for inference) ---
    if avg_val_acc1 > best_acc1:
        best_acc1 = avg_val_acc1
        print(f"🎉 New best model! Saving to 'food101_best_model.pth' with Acc@1: {best_acc1:.2f}%")

        save_path = os.path.join(BASE_DIR, "food101_best_model.pth")  # FIX: persist to Drive
        torch.save(model_state, save_path)

print(f"\n--- Training Finished ---")
print(f"Best Validation Acc@1: {best_acc1:.2f}%")


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.amp import autocast
import timm
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from PIL import Image
from tqdm import tqdm
import os
import numpy as np
import warnings

# --- 1. CONFIGURATION ---
# Make sure these match your training script
MODEL_PATH = "/content/drive/MyDrive/food41/food101_best_model.pth"
CLASSES_PATH = "/content/drive/MyDrive/food41/meta/meta/classes.txt"
IMG_SIZE = 224
NUM_CLASSES = 101
BATCH_SIZE = 128 # Use a large batch size for fast eval
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4

# Define paths
meta_dir = os.path.join(BASE_DIR, "meta/meta")
images_dir = os.path.join(BASE_DIR, "images")
class_names_path = os.path.join(meta_dir, "classes.txt")
test_list_path = os.path.join(meta_dir, "test.txt")
MODEL_PATH = "food101_best_model.pth"

# --- 2. DATASET CLASS (Must be redefined) ---
class Food101Dataset(Dataset):
    def __init__(self, file_list_path, class_list_path, transform=None):
        with open(class_list_path, 'r') as f:
            self.class_names = [line.strip() for line in f.readlines()]
        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}

        with open(file_list_path, 'r') as f:
            self.file_list = [line.strip() for line in f.readlines()]
        self.transform = transform
        print(f"Loaded {len(self.file_list)} image paths.")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_stem = self.file_list[idx]
        class_name = img_stem.split('/')[0]
        label = self.class_to_idx[class_name]
        img_path = os.path.join(images_dir, f"{img_stem}.jpg")
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))
        if self.transform:
            img = self.transform(img)
        return img, label

# --- 3. HELPER FUNCTION & DATALOADER ---
def accuracy(output, target, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

# Use validation transforms for the test set
test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
])

# Load class names for later
with open(class_names_path, 'r') as f:
    class_names = [line.strip() for line in f.readlines()]

test_dataset = Food101Dataset(test_list_path, class_names_path, transform=test_transform)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False
)

# --- 4. LOAD SAVED MODEL ---
print(f"Loading best model from {MODEL_PATH}...")
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)

# Load the state dict (it was saved without the 'module.' prefix)
model.load_state_dict(torch.load(MODEL_PATH))

# Use DataParallel if you have multiple GPUs for evaluation
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for evaluation.")
    model = nn.DataParallel(model)
model.to(DEVICE)
model.eval()

# --- 5. RUN STANDARD EVALUATION ---
print("Running standard evaluation on the test set...")
val_acc1 = 0.0
val_acc5 = 0.0

# Lists to store predictions and labels for confusion matrix
all_targets = []
all_preds = []

# Suppress the GradScaler warning during eval
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            # Move images to CPU, let DataParallel handle scattering
            # Move labels to GPU for loss/accuracy calculation
            labels = labels.to(DEVICE)

            with autocast('cuda', dtype=torch.float16):
                outputs = model(images)

            acc1, acc5 = accuracy(outputs, labels, topk=(1, 5))
            val_acc1 += acc1.item()
            val_acc5 += acc5.item()

            # Store predictions for confusion matrix
            all_targets.extend(labels.cpu().numpy())
            all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())

avg_val_acc1 = val_acc1 / len(test_loader)
avg_val_acc5 = val_acc5 / len(test_loader)

print("\n--- Standard Evaluation Results ---")
print(f"Final Test Acc@1: {avg_val_acc1:.2f}%")
print(f"Final Test Acc@5: {avg_val_acc5:.2f}%")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import pandas as pd

print("--- Confusion Matrix Analysis ---")

# 1. Generate the confusion matrix
cm = confusion_matrix(all_targets, all_preds)

# 2. Plot the full confusion matrix (will be dense!)
plt.figure(figsize=(20, 20))
sns.heatmap(cm, annot=False, cbar=False, xticklabels=False, yticklabels=False)
plt.title("Full Confusion Matrix (101x101)", fontsize=20)
plt.xlabel("Predicted", fontsize=16)
plt.ylabel("True", fontsize=16)
plt.show()

# 3. Find and print the most confused pairs (MUCH more useful)
print("\n--- Top 15 Most Confused Pairs ---")

# Set diagonal to 0 to ignore correct predictions
np.fill_diagonal(cm, 0)

# Find the indices of the largest non-diagonal values
indices = np.argsort(cm.flatten())[-15:]
top_pairs = []

for idx in reversed(indices):
    true_idx, pred_idx = np.unravel_index(idx, cm.shape)
    count = cm[true_idx, pred_idx]
    if count == 0:
        continue

    true_label = class_names[true_idx].replace("_", " ").title()
    pred_label = class_names[pred_idx].replace("_", " ").title()
    top_pairs.append(f"Model confused '{true_label}' with '{pred_label}': {count} times")

for pair in top_pairs:
    print(pair)

# Example output you might see:
# Model confused 'Spaghetti Bolognese' with 'Spaghetti Carbonara': 22 times
# Model confused 'Steak' with 'Filet Mignon': 18 times
# Model confused 'Cheesecake' with 'Strawberry Shortcake': 15 times

In [ ]:
pip install ttach -q

# Test from internet

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
import timm
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from PIL import Image
import requests
from io import BytesIO
import os

# --- 1. CONFIGURATION ---
MODEL_NAME = "vit_base_patch16_224.orig_in21k"
IMG_SIZE = 224
NUM_CLASSES = 101
BASE_DIR = "/content/drive/MyDrive/Dataset"
MODEL_PATH = os.path.join(BASE_DIR, "food101_best_model.pth")  # FIX: load from Drive
CLASSES_PATH = os.path.join(BASE_DIR, "meta/meta/classes.txt")  # FIX: Colab/Drive path
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. LOAD CLASS NAMES ---
try:
    with open(CLASSES_PATH, 'r') as f:
        class_names = [line.strip().replace("_", " ").title() for line in f.readlines()]
except FileNotFoundError:
    print(f"Error: Could not find class names file at {CLASSES_PATH}")
    # Add a fallback if the file isn't found
    class_names = [f"Class {i}" for i in range(NUM_CLASSES)]

# --- 3. DEFINE IMAGE TRANSFORM ---
# This MUST be the same as your validation transform
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)), # Resize to 256x256
    transforms.CenterCrop(IMG_SIZE),                   # Crop to 224x224
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
])

# --- 4. LOAD THE MODEL ---
print(f"Loading model: {MODEL_NAME}")
# Load the base model structure
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)

# Load your saved weights
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))  # FIX
model.to(DEVICE)
model.eval() # Set model to evaluation mode

# --- 5. PREDICTION FUNCTION ---
def predict_image(image_url):
    """
    Downloads, preprocesses, and predicts a single image from a URL.
    """
    try:
        # Download image
        response = requests.get(image_url)
        img = Image.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        print(f"Error loading image from URL: {e}")
        return

    # Preprocess the image
    img_tensor = test_transform(img)
    # Add a batch dimension (C, H, W) -> (B, C, H, W)
    img_tensor = img_tensor.unsqueeze(0).to(DEVICE)

    # Get predictions
    with torch.no_grad():
        outputs = model(img_tensor)
        # Apply softmax to get probabilities
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    # Get Top 5 predictions
    top5_prob, top5_idx = torch.topk(probabilities, 5)

    print("--- Top 5 Predictions ---")
    for i in range(top5_prob.size(0)):
        prob = top5_prob[i].item() * 100
        label = class_names[top5_idx[i].item()]
        print(f"{i+1}. {label:<25} ({prob:.2f}%)")

    # Display the image in the notebook
    plt.imshow(img)
    plt.title(f"Prediction: {class_names[top5_idx[0].item()]}")
    plt.axis('off')
    plt.show()

# --- 6. TEST THE FUNCTION ---
# Find any image on Google, right-click, "Copy Image Address", and paste here
image_url = "tXUdYf0dC59B3p1cvhMDmoGdba7E+9AXHmklOuikYX2E3dQTUNw1ypmuS5qTd9lEjQbNEq9CE1PbauTOZNauUQuaWo0GjbVynjIVok21sGulbzro2/KnoSyG9abkUHquoe6EsaPv7gp280nfp3vIZzJoqFs5yoCPU7t0wohfOn2muEKAahsaMKMCpwlWUUiLbZv3tbrPdmspgbDiPWoXHma1c0xmQ0VoaQzJaoO/RZUSjNdkVyR61G2oUcGTRuuzjo12tQWpOTRSDua6OzmBEbLbnzmKpZbxH5mrb1DUbpA4FVK7bO408V4JyZy4HnUfuJomx066/wofnwKYWujEfHcA9Bk1z0BbFKLFdh5NOl0lhexY+pqT9tRfhRR9KW0OosH6R0Y3PHcEL2Hc/4p8wIG1FgUhv9YeOftQmp1r48WT2zioyyw9lo4prwWFrR9PvULWD5r96rOouuDBYH5GaHfUv50i4PoZqS7Lats+Y+9dtaJ8vvVJOqfzrY17jvTVEGy3NYbyqMKwqsp1m4P3j96Ite0lwc5HrS/GvYeT9D9WM5rsMaV2PaJD8Sj6Yo+zrrL8NtPrQ4M7kg23dxU1rU0N7oxIyPSonaDXcnHsFJjlbs0PctQfQ0Mt7ANGK4YRV4T2SnA5Bre6uNlaZauSJZrKgzWUDgprjdsVENOSZZ6K21yxUckfepNX2Vs4FlfU10ttRwKifWWx3n5ULe6t/CIo0dYyZgoljFAarWlsLx+dCWke54iYX+I/0rd3WpbwnP8R5rgEh05iWIUevP2offat8LJ8z/al1/WM55qA0OXobj7GN7qTHv/Sg7moaeceVQF6ie7StjJE9y/Ak0nu9RBBBYFv4ZAAHrUHW+phBsDqGPmYgc58qqaawvd9wigb+RIwIkuWPpWLPNyfFGzDBRXJjjXdSZOFL58IEiZxIX96mGzVEooUAsFbxXACBnlRJFC6Y2dKDzFtVPvS0szExtthp2rGcCh29prpUiRcQmQQrEhccERnNTS8JDtvsZ2ule7ukm6FLj4QrsMc+IwAaNu31VlRmJLSAdjbQRwGfjIqpavXts3SUWfDLQSZ2iB29R6Gh7Gr9203Va6hG3cjGAxMGR+ua5Jx2tHOnpl1vWyMkYPB7H5GoCKX9O6oNoUIVAbcQJKlc8zw2JgUwTWWyu9fPAP6zVofiU7UlTRKWBqnF6CtN0/dl2CL+J+Qreo1elt/9KjlmiT8qr/UesifiO4YaOAOQf8VXOsdRVtqX0le0MyuSfUfOYpXklPXQygoF4udctODtsE4mYg/MCuEdWI2ErI4P6waqidRtjYqyLcEHPiAGBHm0+dS6TVgFR4omRJ7zxP2qXKUVplEoye0W3TdUuW+CaeaT2iR8XV+o5qoaLqTXF3PiZlY+Eg8TUjAHKmR51pw5uapmfLh47R6EgDibbBh5d6n07xXn2h6i9syCatnTevLcgXMHzqyivBB35LDPeuHrLHGDI8xWmrRF2iMls42Vlb31uiATe8Pma5L1EXqNrnYVxxMblH2dKEG+7z2T+9b09kWRveDc7Dsv+aVazWlyc0rYyQTr+plsD/ApfE5NYq10KTsdGq0wqQLXDsMf4+9CTUVbGSb6IHNC3LtG+HJMxMA+vYYoCdyFxbZQCQSzpnsIUZ+9ReeCKrDI8y9qdaGv788lY4kdzRXs/pUt2LmpkljKqCVwFOQJ5kxMeVCe1XRL1sl4DpkiDlQ2cijv2oWNNYtW7hBKB2xBDsSYDDkQc+WKyyf10aI97I+o68XApBQsrMSGBNyQB8QzKzj6cUKNTcZTu3KfQ8jkyBx2EVCi83NgBLja/cgDMz8WT+BqWzccwpVdu4MGPxEiCDPP0FHSQE5Wci25A8AN0GAGbse8R3kYrpRctwAxKAbtrDORJkCSCDI9akuaoHaAWFyWBc+Iea4wQ3y4rTsVdnueLdzMZCiQDOJJgZ8q5P2jmvIXbsTGzdbK7IRie/JJJ8mmm2nut7rYGAeGBDSYH/TnGYzSTR3ZLkAiAWWTJgdjx+u1HWNQD7thEmSZx4oHhXyB44qc7rfgaFXfsStqNpBDmeWIE8eQ+la1uqW6AdzMxMlT4Yg9o7RS3VdQi6xVAuWEZ447dxQ+muhnkyxI849MVZY/6mSc/A3tM0Fip2pJj68CmmhvG8RtwXjYZAg9hBpNZ1jAwXMxkA8gdiKmta7/AIisi7SpG30Yd80sot2FSSoe6R7oBnwspO71PfinHSNWWJVo4kRVOOuX3hcuSTIb1Y8wOMU86DqS+oUAZM/X/NdBVJOgydxaLI9utQRwaO/ZiW2Su4AEiQds8bo44qO5p4O2RI7AifPj61q5wurM/GXoZ9G66yEBjirbavrcXcv1Fecm1TPo/UGtsM4qsZUTlGy5TWqG/wB+D0rKfkifFlRb2hs/xirH0JR7sahu/wDywfL+KvOfZP2b9/fVWHgXxP8AIdvrxXofWNXwi4AEQOwHAp5uhYWwbXasu3OPzqFVrgYqS0s1IqSBa7gASe1YFpR7T9TWzagkS5gT39KEpcU2GMeToi6h1Z1II8InEwB9TPPpQGq1ynctu4AQoO5j6cgfxTwJquarqJYE7SRBJTcsYgK6qckzS/S3hD7ghySu6VOMxP7nOBXm7l9pdm/UaS6GtzVuWZxfuuVjDBNjZksCBCnnnyrt+rqkvZYgT4zDENI7SMwTniq4moYkbLaoZmJwx4BJmT3Ndam/fKLvZSBAgEjbJJ4EZ9aLiumdyGT9XQzuuqDBO4LlSeceXp86Qa+/bke7beCpyON3cBe3Ez610kIHYKrEgg7gTzAJM8nypRfITdkQzY8wR6eUd6pHGukTlNvsZ3dQrYkmQIJJkcTAB9Iou9eNsK6jxGCCSDAniB3x3pPZ1UgyB4YGO6+c1hvq5AMgYErz8z503D34F5ehrqdQrh2kbiQSYyGHOQfCOalfWBl2qSrFfCQw5wXJ3ekmt9I6C2oD3EkKoJlwwVoGYeDJxwJIpvc9hPdh7t7VWgq7Z2bnweB6NGcmKm3BLY65MR29V7tXJlYYbgRgKeMwZyRzTEawPsuNhQBBAgehPfzNSa72Vukq1i8NTZuz4dwV4GZAbaGAwPP070Ba9ndRe0l5xKBFAh/DuI5TxRBihGKyP6sLlwWypi/uckmZJzz386mt3cyIMeeKL6P7G628CyWwAJks6gfmaaWfYLViDutAH/qLc+YAk5rVKeK65IzxjP0Kn1UwcAxHHOe0VHd1hzuJA7HEg9q11fpGq00+9tsqjAYfAZng+WO9C9G05u3AWyoOZPJ5ya5Qio8vAHJt0Wb2Z6J70i5dVtsyBkEjzngD869DNlDaW3aRWtoRvMYSTJDEZn61WtD1DagOxYDRG7InjvxH51u11JTcG51s3AxwT4RHCsBxMggcYNYZylJuzZCKSVDi87oV/Z2DKG8XKBIjhoG5QPM5mmOp63uKi57vdyLlsCMYyeY8xNVZuqt7wtt/4xMCGBjtuyYC+UVI/VPFNwA3YPjUAbiO5nv27DNLKCkkmFSp6LXa1AYAyGXgFvCTH8PmKIFgcgyPOqUdYAVRWBDjCgYJmDHkflTv2W1pa49s7xDEbWyQR6+RFWwSknT6J5Yxa12PPdmso33VZW2jLYZ0HTCxpd5+K54j8v3RS9jJJNOetvACDgAD6DFJzgVWTtkoqkczRCmKhUmsYE+lKgkvvKo/+0guDbaJSCBnG7mI84/Krvbtmqp/tM07fs9twC225kDMSCJilyL6j43UjzRtUxG3dJ5I5APHPyisN0xO4Bwwgbckec/08qCbUkksQCg8PMN2gwOaj94zAwuR37kTyD581DgWch3pnJI2jePFMAQJ5FT53sgAhgDt4/zSe3rl52lFCw2ck+keeaaezvTn1bFVMbRm5Hwj91f5vrUpQa34GjK9LsC6hqgHYEkwOMHxdsfPvSDqQzMzOf7161ovYnTWlhx79gCSTc2r+BmfvMUVrOiaNSpW1aYqAWO0OEB5B3EDB/igUYZ4w6DLBKfZ44qXAocZBXt/Wn/sv006j3ly7uFi0CzRguwg7FPy5IyAR3Iq73PZzTXDtEWmJMZAkYwiqSsZycRx8+b2quacLp0S37qdnALENliRmd0SfM/ajk/FRklxW2TeJ47cnoJs9Re4qJZtGyFUMLe020ZpiGIwoiMRgVu8Ltx/c3BnaA+xpXa8q095JBEQI+ld6nWWhZtg3Ttu3BMQfDDFogEqAwGcngUdes+61TsTaZSqW7LlkkBVJkgjLAuwBnI8qwPTFc5Se2D6+z+z2/eLvVUUotpEILKCCGc8+vynzxl2/bdLBc77TGWe5IYHAVLcGQSTJJ7Ic1Pc02+0ibxizcts7Fh4pCpO0wQOI7AjOTVWv6HUFnIge78JXdtWVUAFSBGWIGYz5V0Ke7poPNx14LBdnTkKiKyMsK3hi3ukwwJxifFkH51JptaFEEkniYJcxgBTIyBnHbNLuk3nhluQ5ZS5QRIIbbtLnABH5Y9UvUdT/wD07GYlCZQWySCSfA0wCQe4jBEdqrGDkqZqU1XJeS39T6gsbWAuWzAIIggEcQCZ7nccYql9e6Hb0xBsqdhmVJPJ/wCriO1NbPVUTYo8LP8AvAA5IJ2nnBI/GiNWvvLdwYZWAU+LCeRCjHIAkAcd6dXF/oB1JFQXXlAItAK/ikgxK8AHzn8a61PUCbYUncjHPvIJmfEfOeM0puatxNuSCCVaD4McY85FEWj4gG2AgiAeO0ya0ShWyKlehhadWjwTPxQIAkg4phaKFQSdoCsn8s95PYn8KX6vVBd1toWfFhtwGZBGOO8VE3iTBlOdvEcTjuc1N7Wxrq6H9uzbS2hC+JCSCpzglm4kQcRTDp1xwV1AedwaTGBxz68CaSaS8ESLRmJJkSvECe/NEaHUnaA4EljxgEAjMfapSTKwov8A/vv+X71lVzcP4GrK085EeMT0HqjS9B7KI1beM1FcuBVLHgCtjMiNFIEnjufKgn6khIRCC5iJMCJGd3EenNJusdVFxWU3PdqeCpDYyIIjJ5wPKkmt1aGyVA+GGkttfBBGx4JDSMqcYrDk/ENuo9GyGBRXKRZl6z4ty7jtA3EfDj+H6z2jFKdbet3brm2yMxWXDSVYgYIYQyNBHGOcUmt9XlWYKQxwWVyGxPhYzOTyZgipEuq67hutuAJyDE99zNhfkfTFRkn5/cdcX0gLr3sdaJHuGMsPBbDEln2+8ILMuIEiP7VQL++2TbcMDkbTIIg/rNeopql92w2upGN0k+ITztOWyvBmPKq57RaYX7Uvi8olSYBKKIZTJ3c5E+fpVsGRrUhM0E9xKn03Tm/dSyv7x78COSfQV6hptZas2v2awssvhIhQDPxmJySDAMggiqN7B6ZTcdyyCBtG/gzk/WB+dP7+sJYhytx8gNE+7APPEz9MSafPL78V4/2Lhj9bfn/Q7fqboNxK71gArLOyxHJO4QeSBjNLLevubmtxOUjJbYpA5PwjI/e5M0rt6c+8D7mMZEkBST33cn6/Ou7i3QPCyhAGI7yZyxMfhx8qjSRXk2O9Jul7pHiAKocAK8SzgEnIUjHHlwYKs39rMHsFy7Wiu6Co53t68A4keOlKa5PdraN1jcEg7uM4J7mApYc9uKcWC16yiEPtKHGw+HcRMmeJiJ9ajkbvZPNHVoDvJpzcK27Xu2YyUPxHcpJcrEqY8sz5Caa39KotW7l7T+8QACVRGZiAVlwRu79uKW27LAMAjDZu2E7pG0QhIUSefhHarDZREDElbqq6lQrkQcYjG7zgfnSSfRBIJ6HqrDEreUgjb4GWGURuQMQBEQc/LJqXXuGG0Wgtm4/uwVHie2ql95kRBPgg+hHNKdJ1kXnuK2bTW8sQdwB3mCCPF5Z8vrQfU+rC3bVFCgwV3JtBABO2BnIUAmKVR2tDJc9IN1+1LpW2U06FSuDP7wUGFGTA47TzEzXupdS3rNsjk+ILBDd1DdxMnPJn6R2zvJuuoKmW3YATGRAnMgRH14yH1W2iaUhBJO0lmAEAtuGO3xfjFaoRqrLrUHFG7dpAzq8jdNssASU7+Ed8jMV1oWVFJLE48IgzuXbu5iOJ7/Wk1rVSoVyM8ZIk5HNc/wC+AhKAeahQBMtHfvVeMnrsS4p2KvaYxqGg4IUn5xB+dRaK27mbas7LJhVLdo4UVfuleyquo1Got+9YKqpbLAcHkxl2zMeQ4qxr1kWxbt20FhT4VLK3iPJCKsIsQc5iKf5kopUD4W5Wed2uk6uCg0txmY8lGECJIloAgVmosXE/5tl7YWeVMehn1Fen6nqbzCzk/DKltuJZdqsWxJ9D3FD6nrkBQMqCpJIWIyGDsQZBxM7T9jUlNyWijhx7KB0/UhCFPAUxIkZgnmmGmvb3PhBBIyeQBHFWbqvRrF0eFVRiB4kUjnjdbHb7Tjjmk2i6S1q4VaCeZHwkdivpXY4qctgm3COhzurKm9zWq2mUt+qbxt86rvtV1IC21vxDBk8TiQR6c59KsPUUi6a839tdUV1DyRt8CkHkDaDuEcxJNSz3xpD4auxHpWhPDMByZ7gSY3E5kDH4UbavWl3TuJO2T8Sw2SCBEZ/R4oRNM+NslWETHIwzDb6HEz25qO8F922zAI3ENidpAEHuee84rLSky+0htrdEssQ4uKgRt22BFwYAyNxJ7gdpxQXuwSq5D7WLLkHaMTnBGe34VDbcSIwo2eLkGOTI+eMdqi6hrYaCPDBIYySRA5BgxEniuUX0FtJDFtUwRW2gwDhhkyZkgCCRM+QpZquoEeKPEIcYEE9skTEQO/FDpqHiMRmGEgDtBnj/ABQ16/gifh9fuIjPb7U6hsXlo101wu5idoaWnbJAgyY+9HWL7BcODnJ4ERA/pz5ml2iViqM7HaTcC+Ug8R5QWxNFLZ2oJBAafkfw7Y+9UyL7CQb46JNTc8RQDaMLA3NEYImcH+1R6YkkwTHw+LgceI+v25ru5eMoBzAxn4uDI/pQ2u1yltwPiJacFRkRM8+QpVbVDOk7GXvEUblJZlYb8AlRtUYec5J/UU09n+suLioWbKhRBkQefpIme2KpC65rbMwMgwp7Hg/jmnGmvzsKERHIkEceWQZEZHP3rsmL67OjNS0Xi77y1sa65NsyT7oQ6wCfhX4uQDGTPEVJ7NC1cIe1fF1BuHG0gnneScEgnPrSTpvVjK7yxRBu5PhIAkiM/YeXrWtfq3ZvCTsGQRgchoHfH9ayOEuvP7CrFv8AQsOmFuyHthkFtQVKM3/EfkKNyr4wAeTkwZqra6dRtKsgCsBEGQvA2keg7+Q+u31Vy4AG5UggEzjzWBg5FRadZYKyukNnMQO+8QYM8DP9KpCLjt9lIwUdDiydmCpUAehUxHb94H6d6rXtBqAqm06y54I4wQAfWAI+tNPeFwg7y/hkCYURyJGJ9PxqjdQ1zXnL7TwAIzAHr65NXwwbkDLLijGvwRyI88g+g8qZdDQPc97sDR8K/CJAkkkcwKr3iYgZJ4Ap90cXGD7QSEQBgo3eAeYGTJz9K05Y1HRnxO5F2Xqj3W3+7ZBaIDXLa2wtuFxsBwXjyGcSM0PpuosAX3bWZHIUqdhkgElAwWAOPKc0vt6sMIZWgKUABja4kTtOINb0jWnH/GU7yCDiNxCwNxiD6jOIHNY3HW0a1KnpjHpesZ+SbZA8S2ji4phmASfD4fvySaIvyt1isLbBVlSCFUASZaSf4scZpYuoKkMACTlQP3IMlh2ggx9K1p9arM4aHwNyzAMk+ESACBmc/KaFO20FtUkx1pepJGMlkMA4Ix/5ZHMbpj0phooB2ly4MlWbDKOdr+ZExIxM1XLOrAAgxG50ACwQDtOIMNGD8ge9NXu7SqkysyQcQYkkDzEqftSpcXr+fz/oXtbLR7g1lA/76+X3H9q3Wv5l6M3w/qXLr1uG3diK8v8Ab63/AMVeIuKPLBTOZ58q9a1lv3lj/qSQfp/ivMfbawDaII4yD3B/tVckOSJY5UVGwbgBhmG3LhcL44G3BMCR9z8q6uXDIESgwynsSeRPGO/fFD2tSpXJMkx2BHrMEfh5UTcvqoDgZwp5YmJyZ75j6Vi/ujT/AGONZZCqQWIEDbPDCBB8+BW9TqCtuQsgKqgDjGAD6zJz511fumIc7lWMk/CAMRjt5UNdumRtcATJHOD39CP7Vy8Wc+2Q6wHaoMgFRJAx55ng0v6gIJGdyhQSQBk8GPL+1Fa69uMkgrAweeR6fqaV6q+zkKfE0BQZ4UcCfl86tii2TnJDvRWN+ktj95SXXsJJODHmDWtPYvXgUto9xhA2LJ2g8SeB5ST2pt7L9Ja8MqRbReAQu6OwLY+eRyKvlnSW7aLbU+7LnhYids7WuCZExJEsc9ors2SPKl2HFB1b6KLp/YO8AGv6hU81XLKCCQu6fiJEQA351KPYfT7X3XLm5ZxDboJwwVgBJyPI+lXAdasMXC+8tvbwXKgBuV8FwkAKeROT9JoPT399j3vvWKyE928uG2zLm4x5BxJxjtWd5ciLrHA866x7IPaUFWwQDDiDnvIxgHPyNVu3vQ7hI9ex7favQuoqsAwVWCcMTtM5BJO2e4+Y9IrHUtFhsMTMqfTkkxyPI+lasWZvUjPlxJO4nOh6oHIW4NpnBWQM+vIPNNumbRhrg2/iOQP5u1VzRaeCfz/XFF61fCB+8ePkOTVJYk1pk45WntDSxqIlyyrONoJBxEwPKak1fUSqhlJYTwcFmIExGcGYjzP1r6aIBjkQADODEZyM/o0y0ZlCoIJMngfON3Y5I+lRlGK2WjJ9EqalmbcTB2iBmBM5aY45/CrD01tuSxBVR4QIEkYmBhfn5cGlOjsEwWVpKwomfWTnHfnn604W+QdxZScQ+QTuObYM/EBkHjHbmoTfhFIew/8A3Zpr4Lm3b3MCd6AK2MSsYmIx+dUfqfR7uiaQ5eydsOgwcghWg+Aj8avPT2a2QCGJz8RJ7HtPMNAJIrrqd5L6e7dyBcUBsyYghSJWQQQMn+EihDI4yrtDSgpK/JQbOs8YDGQwmTHccmOQp7Ufpb8KytDEYhTgqcAiZjtOM0gu2TZuXLdwEuuFP7vmGMj4SM/X5066bfODAG4j6hRAJj1rRkikrRCEm3QUdqxHwKMq2DkZ8QGIG38ane9EFrQDg7SQFCtLTljzH1z5TQy3PEz3DuJOCAQBAMEgYBIgzHcY8pTtQqp3Hd8R5ZWxt8JxImR8o70jSKJugrRPDTtBXaY4A44JwMkR3nzo69qnFkq5mFgE8ltuT88/KAeaXe6KsqfuDuSTII+3NTJq2llILI8EDDZMwZ8pJxHYVOW3odXWze21/wBP2b+1ZT7/AHNc/wDUH3b+1ZR+KXp/5QvOH6f4PUNHqArwfhbwn59jVd9suj4bGDxTm8jeVGWWF9DaaNw+E+denJHnRZ88dZ6OymVmO4oPS6vBDGCOxOeIOO9eq9e6LtcgiqT1b2eBJMVGcFIrGbQhu3veLjAgycDg8kdyRH2qJ9QoiCMcxyf1/Wpb/RXUEAmJ70MvTLk8mkWKhnOyHWaolvCJ+gz/AEorofSTcugGRJkwJIHJgDvRGm6ZBk5NPdIiJZYFQCSDuMmcyFkZUEyJg/hRm/jhoMF8k9lo0l1IGnQKrBT4SNxAAJJMjtiZDRuzQd7qDG37whgw3Bm27gRMSqHwqdoH7vfGZpQ94WrZ90vjuEAypYMpkzkknJYSeRGAZqbV3ELsThcPkgEmQNoWSVwxIzwKwJbNkmEprXvKfdBQqj4TnczEMCJjacERBxQmt1zMM7TbM72Ih1hceMwOc8+fypdqbllSVWQRDHa/OAYnEMM94MGjrqLt8O8icbyCbnpEgDE5/OqNJPYiba0c6mwbZFxHJdlhjkMMcGMEEeQ5BoB9GCowQSVbghsiJ5zxHemDqNxAO0+7JVP4YxtbbMjj18VQ3S7bHOIkAwREEA+GYgkA+sUE2GS3ZXbgi4JG0Hw9s8kGR+s0LfE3WzgQmOfkPWfzozrd4kls98kcMrDvgEZB+1BdOBZZyWLmT5T+Ez39a1xl/wCZlkvuMOnIRJjIgwIYknk48sCKMAYbJVSZAAJjn5d5qG3uVzsILMQ3PG1ZMEeuB/mpbOqLbWbaNxzI2zkCTPGe8jiou27LLWgu04S4AYRTkDaMYI8szM586YabTTMnMbjCn4RhnHpA7T60Dd08NaJYHggwOAZAM4JmmLdS3D3nvTKjZCARA+MMZx5n5jyqMr/pKRpdkOhunaSrQ3J7rG4xtYZkiBHfNH32HxlwoChW8+VGDGYOM+Q8qV2XVgJXZLksM5x4WAk/l39KNkAhCkrubcQB4cAz67iQfoaE6bDBNIqntXbC3UcHdulZkcCNkgejEfIUHoL8MAT35gDPY/o96Y+2CKQNo4uNGeBxxzHhEZpIl/A4BiMj8c962QXLGjLN8ZssemCoQ0kHdjny5ABgGRWr8s3ZpyS0kkkAxnM0ssdQAXxDcezA7CIxzmR86jTWyZE9sEyPSpcJXZTnEb3tRgGd0QfRZ5IHPAx86I6Cu657yJIJAMz39fL+tKtLo2uECCFPI88z9K9A9kvZx7zbVEKI3NGF/ufSq48VbZOeSyXcfOsq+/8AhCx/G/4VlWpkuR3dvHgZNQal/dRzIyY5mj9LZOXP0/vSrq1zkT+FNklQkENbtu3q7YBIDxhuzf59KpXUenFGKOPP5GpdLr3svIO5e6/1HkattjV2dVbh8jgN+8p8m8qRSUh3Fo811HTB8xQF3p4GIirz1nodyzkZU8MMj6+VJbtod+00wCtpoaE6hZyRxtUP3zBMxHeBMelWf9lAyO8STmcRSj2gtBU94P3RBxypgcjy+1SypuOiuJpS2JdNcVXYKBJIDFiTmCANoxHGRzRVmztVWXML4wTEzncMiRM4AxQGldZII2x3I+u4+eaJVR8fM4lhKkHzjgz+dZJLZpjojt2hJblZO0gYWW3Yfk4G2DPnms0Vv/giSVY7RDHws3bEGCAfMcQaICndCsCWAx+6e2ACZz3FAacsrlSchyZaNvOCYmBzyKK2uzun0G2QqTu3F1KgSI3T4m3d42qO+OOK5uXwbbGZAK+RCnxAFYEgf48xUJuFlglsQSR3Mjd8hECI4rlNQqxuIZshpOIC/Fx5gD0rnHVgT3Qr64VA2Ybyg9zkCYH6FCAHaVEAljzznxD58RXerALHaIG4mMmCcnJ/XNbeYBjjy8v1BrVCH0/czTkuQ2sgBrbq42lVkZOQPhbyOSPLPNT6iwm1dxQIxEhSGIGSY9R37UJpdTKY2wQeYjGSWx4TgZ+td2NYXG3ZBAmCYHi7AjvH0rMrTv0aNNUFIVVSylmfhTAVABwZyc4x3qZgWJXaSctGCJxLCIgZzyMUNo9RaFzZfU7PJVDFiAfiPG2YPah7W+AAS3ECfDtyYEZiOw4oUCxs2uAYFZ2mIJO4wo8Ix/XE/WpdKy7iT4QASrTkspkj55NKdRcICbVjaBLSsgHOdpO6J7jnzrrT6tzs2hd0wwzBBEcHtOfTNJwtaKKVPZu/pRcJiYViJPfPpj7UI/RhVk0GiCIFgAenA8v6Cj10IxIJ/XfzrfBcYpGKTt2UxPZ+c/PtTHS9DUQTH6/L/FXbpvQbl/FtJHdjhB8yfy5q49J9nLOnh3i5c8yPCp/6VP5n8KcQqvsx7FM8XLn/AA7f/wBm+QPA9TV6Ny3ZQW7ahVHAH6yfU1B1DqcY7+X96XpdJz3ipyyJaQ8Y32G/tb+lZQ28/wAQrKnzfsfih5cSFCjAAqudTtZ5qz3NLOSSfrFKdZoQTMH5T/Wqz2TiVHVWjmlaaq7Zbekg9/I+hHerlqdCYpVqOnTUmqKpjP2f9q0ueBoVjgo3wt/Kf6UbruhWrubR2N/A3H0NUHqnSj2FMumdW1FhRvm6nkfjA9Cfi+v3qkZ+xHD0Sa/ply0drKQflAPy7H6VXupuROJ/WQc/qa9D6T7UWbw2K4J7235/0n8xWdR9ntLfHwm2x7oZH+k/0p1T6EaaPC9Tf2NxKzkHyiI+XHftUdjqrLCKQBIOciZyeJGK9D61/s0umTauW7noZRvscfjVE6n7G62wfFp7kfy71/1LIFTlhRWOVo1+2IwBko8gAyRwZJHznz7TU/7TBkfEu7iPFwN2arwDoxlSD9ZnP9zRNu67fun8alLEykcqHS30Fp90o3gCqsGROSeR+hSu4DcI24nmSCQPL0H1ru1orjkM0g4/zjinGm0AXgU0MTW2JPIn0LbeixEVu/oiBP6+vnT23pDnz/Ly5rq9pifl+u1aCJSEQ2zIkCZkc/I+lEJqByB24OY/UeXFO9Z0WZxHypU3RGBwYqcoWUU6OBfLZ2xAyMmTESJnPrWW9TckKAT4ZBk9sQf1xRuj9jdW5/4dtzPkpA+5xVi6d/sr1bkG5cFsQeWkgHkQoI/Gl+IPyFNsK5O0QSYx5f2qz9J0Hu1GPEcHPA8h5mr90T/ZvprA8dxnPeIUf1P41Z9FpdPYj3VtFgcxLenjOaeMK2JKd6Kd0r2Xv3QJTavm8riPL4j9qtWg9m9PZzcJut64T/RMn6mitR1GMkhR50LeukxDIARO5mn7IPi+9M2kKk2ML/UBG1YCj6AD0AwKEe6zDw/c8/QUIty2uW33D3MAD6LRdvX2AJBIHeVOKm5t9DqKQGdFBJ5JqW1aimlnZdTdbZXHYg1EdIftU3ChuQH+z+tZRXujWUKDyHzKT3qC9Yqn3fbx5xaH3/xWz7f+HNrPzx+VV+WHsn8ci0fsgiKD1HTh2qtp1C/qBudiA3CpjHb1NN9H0I/Fddv5dx/E0OXLpDca7ZBqumE9vxFK9XcsWABeuhSZgBWc/OEBgVZf92W1naq55Pc/U1E+kH8IIrnFnKSKT1T2btalfeWGW4BmUPiB9RyppcNV1DSfC3vUH7t2Sfo2G+5NX1+mpMhQD5xB+4rhtK3G4keTww/Gl2hrTK70r2/nw3rFy0fMeNPwz+FWnR9at3BKOp/+p/04NAvogP8Ay7f+n+s1E2kYfDbtAfyT+Zo85I7imO7ptOIdFb+ZVP5igbnQdC3OntZ8l2/9pqC3ZPcAfyll/CYrCWHCT6lv7LTfIxeCOz7L6L/0iPk7/wBTWx7K6P8Ahf8A1n+1b96Y4YfY/wBqjua6O/3x/Wj8gOBL/wCFtJ/C5+b/AOKkT2d0g/8ALJ+bt/ek9z2kVeWSP5pP2FR/+KbZkqWJ8thz9Zo/IjuDH69J0o4sJ9ZP50RbFtPgtovyUD8qpN72ovn4UVfKQSaX3+tasmd5A8gg/tNd8iBwZ6O+soK/1hAYNxQfKc/bmqgnWLxAL2rR+YafsG+dEaS6u6bdpLbEZIkn6TxSvKMsY91PWFVdzEqvYkEE/wAq/EftSvVdR1FyPcgW153OJJ/+I4+9EabQgn+Jj3OTj1Nc6/rGn05Kk+9uDGxOx/6m7fjQucujmoxB9N0q5cYNec3COMQo+SjFWnRdOCgeGqG/tpfZgLapbTOBlv8AWccenaoR1C+p99bvXGEgsNxkfzDv+Vc8fHbOU+Wkej63SiIqvdbb3Vpj+80qo9T3+gqwpqwbAuuRG2Se1VDqRN5w5+HIUTgD1APNOsdsnKdIQdOa7a+B2AHMGCKsGl67qQBF0n0YA94OY866/ZAskgrI5A58jHcViaVYJIx9InnuOcVXiStkv+99Z/6g/wBKf3rKm/Zbf8J+/wDispuJ1srd5K50ul95cVPMgf3ou/aimnsVot98uf3Bj5nH5TXmxjbo3t0rLf07QJbAxkfh8qJuH1Fd30ng/wBaE3mtnWjN2adqHN6iHzQd5DSNjIy5qh3oS7rorLox96BvITU22OkSP1pF5/Kg7/tLHwWyfnj+5qNtGDQ93Q+QoWxqRl72oufu21+ZJNRWup6q5+8qj0XP0ma2Omkn+9E27G39flXWw0R/s1x/iuPHkDH5VyOkJ3Wf5pY/jTvSW9wqd7Kou52VFE5Y8+gHJPyo0CyuXukquURR8hWWdNOasGm1+mc7RcOe7Kyj7nitXtBs8RZQs8kgD70K9Av2LH0XhxQ505inFvXaYyo1FrcMEbxz5V0dMHzbdGHfawMUziwckIbWm86m01kKab3elkZBEecgChbl+zazu96/ZFOJ9W8q5Qk9I5zS2wb2j1xs2RbtmLtwZI5S35+k1TdLojEHyzx5/KT2qxXd11zcYElu44jso+0RXX7EImcycdhif0K3QhxVGGc+UrE1vp5EQpjv68du/BH0qW3o2B8AKjdiMzOY2+WKsNqweTwCAM+Xn6Gp1T4uJMRkST9eBn8KPEF+jLLE2ltbTsBLYPfkY8vT0qS1ZQnaZ3DkCAfIY781lucSfmZnA7cjHp8qLAI8cc/CZwsnPHf8PtR4paBduza2I8PfyEycZmeIrfuRLDMYPcjj+H9c1yLYbdnnbkkjPBx2z38qLsW8iWzkef0PMTmKFDWc+7Hkv/0/vWUR+yj+Eff/ADWUA7Kh1H+v9aeew/wXfmn5VqsrzsX5zfP8patP/WhNX8X68qysrQ+iC7IdPy36860/JrKyk8Dgdyg7/wDesrKmxkc6Xn71O9ZWVwQK5Ubf1rdZXBD+l8Ur9tPjsfyn86ysrpfkZy/MDWeP151VPbLkfJPzesrKb8P2Jn6K03A+Y/Om/RuD8v61lZW0xrse6z/lj6f9pqXpnw/rzFZWVSPROfY56dwf5v60Yn/Kb9fw1lZQ8h8A1z4P/in5iu24X5f/AIrdZTiIYXf+Q/8A7h/7TQ934F/+H5NWVlKhpDmx/wAu3/7Z/wC2s0Pw/U//AJrKylHQFWVlZVCR/9k="
# image_url = "https://www.foodandwine.com/thmb/9-a-rF6dOce-1B1V5j-Wp-E5vI0=/1500x0/filters:no_upscale():max_bytes(150000):strip_icc()/steak-filet-mignon-ft-recipe0817-64057c66c35c45509a7b71f54b4f603a.jpg"
# image_url = "https://sallysbakingaddiction.com/wp-content/uploads/2013/04/triple-chocolate-cake-4.jpg"

predict_image(image_url)

# Fine Tuning : Arabic Dishes

In [ ]:
import os

# This path comes from your Kaggle URL: .../datasets/araraltawil/arabic-food-101
# (Kaggle shortens it to just the 'slug')
data_dir = "/kaggle/input/arabic-food-101/Arabic Food 101 V3/Images"

print(f"Checking directory: {data_dir}")

folders = []
if os.path.exists(data_dir):
    # Check if the root directory itself contains class folders
    folders = [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]

# If 'folders' is empty, the classes might be one level deeper
if not folders:
    # Try to check common subdirectories like 'data' or 'images'
    common_dirs = ['data', 'images', 'train', 'Arabic Food 101']
    for sub_dir in common_dirs:
        check_path = os.path.join(data_dir, sub_dir)
        if os.path.exists(check_path) and os.path.isdir(check_path):
            print(f"\nChecking inside {check_path}...")
            folders = [f for f in os.listdir(check_path) if os.path.isdir(os.path.join(check_path, f))]
            if folders:
                data_dir = check_path # This is our new base directory
                break

if folders:
    print(f"\n--- Found {len(folders)} Classes ---")
    folders.sort()
    for i, class_name in enumerate(folders):
        print(f"{i+1}. {class_name}")
    print(f"\nOur new data directory will be: {data_dir}")
else:
    print("\nCould not automatically find the class folders.")
    print(f"Please manually check the contents of {data_dir} and its subfolders.")

## Imports, Config, and Merging Datasets

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler
from torch.amp import autocast
import timm
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
import numpy as np
import random
import warnings

# --- 1. CONFIGURATION ---
print("--- Configuration ---")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# --- Paths ---
BASE_DIR_101 = "/content/drive/MyDrive/Dataset"
NEW_DATA_DIR = "/kaggle/input/arabic-food-101/Arabic Food 101 V3/Images"
MODEL_PATH = "food101_best_model.pth" # Our SOTA model
FINAL_MODEL_SAVE_PATH = "food_finetuned_model.pth"

# --- Model Config ---
MODEL_NAME = "vit_base_patch16_224.orig_in21k"
IMG_SIZE = 224

# --- Data Config ---
# *** "AW SNAP" FIX: Set this to 2. If it still crashes, set to 1 or 0. ***
NUM_WORKERS = 3
PIN_MEMORY = False

# --- Stage 1 Config (Train Head) ---
STAGE1_EPOCHS = 8
STAGE1_LR = 1e-3
STAGE1_BATCH_SIZE = 128

# --- Stage 2 Config (Fine-Tune All) ---
STAGE2_EPOCHS = 10
STAGE2_LR = 1e-6 # Very low learning rate
STAGE2_BATCH_SIZE = 64 # Smaller batch size for full model backprop

# --- 2. DATA PREPARATION (MERGE DATASETS) ---
print("\n--- Data Preparation ---")

# --- Step A: Load original 101 classes ---
class_names_101_path = os.path.join(BASE_DIR_101, "meta/meta", "classes.txt")
with open(class_names_101_path, 'r') as f:
    original_classes = [line.strip() for line in f.readlines()]
print(f"Loaded {len(original_classes)} original classes.")

# --- Step B: Scan new 20 classes ---
new_class_folders = [d for d in os.listdir(NEW_DATA_DIR) if os.path.isdir(os.path.join(NEW_DATA_DIR, d))]
new_classes_cleaned = [name.lower().replace(" ", "_") for name in new_class_folders]
print(f"Found {len(new_class_folders)} new class folders.")

# --- Step C: Create new master list ---
master_class_list = list(original_classes)
unique_new_classes = 0
for new_class in new_classes_cleaned:
    if new_class not in master_class_list:
        master_class_list.append(new_class)
        unique_new_classes += 1

master_class_list.sort()
NUM_TOTAL_CLASSES = len(master_class_list) # Should be 119
master_class_to_idx = {name: idx for idx, name in enumerate(master_class_list)}
print(f"Total unique classes: {NUM_TOTAL_CLASSES} ({len(original_classes)} original + {unique_new_classes} new)")

# --- Step D: Create combined Train/Val image lists ---
train_image_list = []
val_image_list = []

# D.1: Process original Food-101
print("Processing original Food-101 images...")
train_list_101_path = os.path.join(BASE_DIR_101, "meta/meta", "train.txt")
with open(train_list_101_path, 'r') as f:
    for line in f.readlines():
        line = line.strip()
        class_name = line.split('/')[0]
        img_path = os.path.join(BASE_DIR_101, "images", f"{line}.jpg")
        new_label = master_class_to_idx[class_name]
        train_image_list.append((img_path, new_label))

test_list_101_path = os.path.join(BASE_DIR_101, "meta/meta", "test.txt")
with open(test_list_101_path, 'r') as f:
    for line in f.readlines():
        line = line.strip()
        class_name = line.split('/')[0]
        img_path = os.path.join(BASE_DIR_101, "images", f"{line}.jpg")
        new_label = master_class_to_idx[class_name]
        val_image_list.append((img_path, new_label))

print(f"Original: {len(train_image_list)} train, {len(val_image_list)} val images.")

# D.2: Process new Arabic Food V3
print("Processing new Arabic Food V3 images...")
new_images_added_train = 0
new_images_added_val = 0
for i, folder_name in enumerate(new_class_folders):
    class_name_cleaned = new_classes_cleaned[i]
    new_label = master_class_to_idx[class_name_cleaned]

    class_path = os.path.join(NEW_DATA_DIR, folder_name)
    images = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)

    split_idx = int(len(images) * 0.8)

    for j, img_path in enumerate(images):
        if j < split_idx:
            train_image_list.append((img_path, new_label))
            new_images_added_train += 1
        else:
            val_image_list.append((img_path, new_label))
            new_images_added_val += 1

print(f"New: {new_images_added_train} train, {new_images_added_val} val images added.")
print(f"TOTAL: {len(train_image_list)} train, {len(val_image_list)} val images.")

# --- 3. DATASET & DATALOADERS ---
print("\n--- Creating Datasets and DataLoaders ---")

class FineTuneDataset(Dataset):
    def __init__(self, image_list, transform=None):
        self.image_list = image_list
        self.transform = transform

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_path, label = self.image_list[idx]
        try:
            img = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, FileNotFoundError, Exception) as e:
            print(f"Warning: Error loading {img_path}, returning dummy image. Error: {e}")
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))

        if self.transform:
            img = self.transform(img)

        return img, label

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.08, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
    ]),
    'val': transforms.Compose([
        transforms.Resize(IMG_SIZE + 32),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
    ]),
}

train_dataset = FineTuneDataset(train_image_list, transform=data_transforms['train'])
val_dataset = FineTuneDataset(val_image_list, transform=data_transforms['val'])

# Dataloader for Stage 1
train_loader = DataLoader(
    train_dataset, batch_size=STAGE1_BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_dataset, batch_size=STAGE1_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

print("Data preparation complete.")

In [ ]:
# --- 4. HELPER FUNCTIONS ---
FINETUNE_CHECKPOINT_PATH = "finetune_checkpoint.pth"

def accuracy(output, target, topk=(1, 5)):
    """Computes the accuracy over the k top predictions for the specified values of k"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            # *** SYNTAX FIX: .reshape(-1) ***
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

def run_validation(model, loader, loss_fn):
    """Helper function to run a validation epoch."""
    model.eval()
    val_loss = 0.0
    val_acc1 = 0.0
    val_acc5 = 0.0

    val_loop = tqdm(loader, desc="           [Val]", leave=False)
    with torch.no_grad():
        for images, labels in val_loop:
            labels = labels.to(DEVICE, non_blocking=True)
            images = images.to(DEVICE, non_blocking=True)
            with autocast('cuda', dtype=torch.float16):
                outputs = model(images)
                loss = loss_fn(outputs, labels)

            val_loss += loss.item()
            acc1, acc5 = accuracy(outputs, labels, topk=(1, 5))
            val_acc1 += acc1.item()
            val_acc5 += acc5.item()

    return val_loss / len(loader), val_acc1 / len(loader), val_acc5 / len(loader)

# Suppress warnings
warnings.filterwarnings("ignore", message=".*`torch.cuda.amp.GradScaler(args...)` is deprecated.*")

# --- 5. LOAD MODEL & CHECKPOINT ---
print("\n--- Loading Model & Checkpoint ---")

# --- Initialize defaults ---
current_stage = 1
start_epoch = 0
best_acc1 = 0.0
optimizer_state = None
scheduler_state = None

# Load the base model
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=101) # 101 classes first
# Load our SOTA weights
model.load_state_dict(torch.load(MODEL_PATH))

# Replace the head with a new, uninitialized layer
num_in_features = model.head.in_features
model.head = nn.Linear(num_in_features, NUM_TOTAL_CLASSES) # Now 119 classes
print(f"Model head replaced with new {NUM_TOTAL_CLASSES}-class layer.")

# *** GRADSCALER FIX: Call with no args ***
scaler = GradScaler()

# --- Check for fine-tuning checkpoint ---
if os.path.exists(FINETUNE_CHECKPOINT_PATH):
    print(f"Fine-tune checkpoint found! Loading from {FINETUNE_CHECKPOINT_PATH}...")
    checkpoint = torch.load(FINETUNE_CHECKPOINT_PATH, map_location='cpu')

    model.load_state_dict(checkpoint['model_state_dict'])
    current_stage = checkpoint['stage']
    start_epoch = checkpoint['epoch']
    best_acc1 = checkpoint['best_acc1']
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    optimizer_state = checkpoint['optimizer_state_dict']
    scheduler_state = checkpoint.get('scheduler_state_dict', None)

    print(f"Successfully loaded. Resuming at STAGE {current_stage}, EPOCH {start_epoch + 1}")
else:
    print("No fine-tune checkpoint found. Starting from scratch.")


# --- 6. STAGE 1: TRAIN THE NEW HEAD ---
if current_stage == 1:
    print(f"\n--- Starting STAGE 1: Training New Head (Epochs {start_epoch+1}-{STAGE1_EPOCHS}) ---")

    for param in model.parameters():
        param.requires_grad = False
    for param in model.head.parameters():
        param.requires_grad = True

    # *** DATAPARALLEL FIX: Define optimizer BEFORE wrapping ***
    optimizer = optim.AdamW(model.head.parameters(), lr=STAGE1_LR)

    if optimizer_state:
        print("Loading optimizer state for Stage 1...")
        optimizer.load_state_dict(optimizer_state)

    model.to(DEVICE)

    # *** OPTIMIZER STATE FIX: Move state to GPU ***
    print("Moving optimizer state to GPU (if resuming)...")
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(DEVICE)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for Stage 1.")
        model = nn.DataParallel(model)

    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    for epoch in range(start_epoch, STAGE1_EPOCHS):
        model.train()
        train_loss = 0.0
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{STAGE1_EPOCHS} [S1 Train]", leave=False)

        for images, labels in train_loop:
            labels_mixed = labels.to(DEVICE, non_blocking=True)

            with autocast('cuda', dtype=torch.float16):
                outputs = model(images)
                loss = loss_fn(outputs, labels_mixed)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            train_loss += loss.item()
            train_loop.set_postfix(loss=loss.item())

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss, avg_val_acc1, avg_val_acc5 = run_validation(model, val_loader, loss_fn)

        print(f"Epoch {epoch+1}/{STAGE1_EPOCHS} [S1] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc@1: {avg_val_acc1:.2f}%")

        # --- Save Checkpoint ---
        model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        checkpoint = {
            'epoch': epoch + 1, 'stage': 1,
            'model_state_dict': model_state,
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_acc1': best_acc1,
        }
        torch.save(checkpoint, FINETUNE_CHECKPOINT_PATH)

    print("--- Stage 1 Finished. ---")
    current_stage = 2
    start_epoch = 0
    optimizer_state = None
else:
    print("--- Skipping STAGE 1 (already completed). ---")


# --- 7. STAGE 2: FINE-TUNE ENTIRE MODEL ---
if current_stage == 2:
    print(f"\n--- Starting STAGE 2: Fine-Tuning All Layers (Epochs {start_epoch+1}-{STAGE2_EPOCHS}) ---")

    base_model = model.module if isinstance(model, nn.DataParallel) else model

    print("Unfreezing all model layers...")
    for param in base_model.parameters():
        param.requires_grad = True

    # Re-create DataLoaders with correct config
    train_loader_s2 = DataLoader(
        train_dataset, batch_size=STAGE2_BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )
    val_loader_s2 = DataLoader(
        val_dataset, batch_size=STAGE2_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )

    optimizer = optim.AdamW(base_model.parameters(), lr=STAGE2_LR, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STAGE2_EPOCHS - start_epoch)

    if optimizer_state:
        print("Loading optimizer state for Stage 2...")
        optimizer.load_state_dict(optimizer_state)
    if scheduler_state:
        print("Loading scheduler state for Stage 2...")
        scheduler.load_state_dict(scheduler_state)

    # *** OPTIMIZER STATE FIX: Move state to GPU ***
    print("Moving optimizer state to GPU (if resuming)...")
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(DEVICE)

    if not isinstance(model, nn.DataParallel):
        model.to(DEVICE)

    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    for epoch in range(start_epoch, STAGE2_EPOCHS):
        model.train()
        train_loss = 0.0
        train_loop_s2 = tqdm(train_loader_s2, desc=f"Epoch {epoch+1}/{STAGE2_EPOCHS} [S2 Train]", leave=False)

        for images, labels in train_loop_s2:
            labels_mixed = labels.to(DEVICE, non_blocking=True)
            images = images.to(DEVICE, non_blocking=True)
            with autocast('cuda', dtype=torch.float16):
                outputs = model(images)
                loss = loss_fn(outputs, labels_mixed)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            train_loss += loss.item()
            # *** SYNTAX FIX: Removed bad formatting ***
            train_loop_s2.set_postfix(loss=loss.item(), lr=optimizer.param_groups[0]['lr'])

        avg_train_loss = train_loss / len(train_loader_s2)
        avg_val_loss, avg_val_acc1, avg_val_acc5 = run_validation(model, val_loader_s2, loss_fn)

        scheduler.step()

        print(f"Epoch {epoch+1}/{STAGE2_EPOCHS} [S2] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc@1: {avg_val_acc1:.2f}%")

        if avg_val_acc1 > best_acc1:
            best_acc1 = avg_val_acc1
            print(f"🎉 New best model! Saving to '{FINAL_MODEL_SAVE_PATH}' with Acc@1: {best_acc1:.2f}%")
            model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(model_state, FINAL_MODEL_SAVE_PATH)

        # --- Save Checkpoint ---
        model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        checkpoint = {
            'epoch': epoch + 1, 'stage': 2,
            'model_state_dict': model_state,
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_acc1': best_acc1,
        }
        torch.save(checkpoint, FINETUNE_CHECKPOINT_PATH)
else:
    print("--- Skipping STAGE 2 (waiting for Stage 1 to complete). ---")


print(f"\n--- Fine-Tuning Finished ---")
print(f"Best Fine-Tuned Validation Acc@1: {best_acc1:.2f}%")

## Evaluation Test

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
import timm
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from PIL import Image, UnidentifiedImageError
import requests
from io import BytesIO
import os
import matplotlib.pyplot as plt

# --- 1. CONFIGURATION ---
print("--- Loading Configuration ---")
MODEL_NAME = "vit_base_patch16_224.orig_in21k"
IMG_SIZE = 224
FINAL_MODEL_PATH = "food_finetuned_model.pth" # Our NEW finetuned model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Paths for re-building the class list ---
BASE_DIR_101 = "/content/drive/MyDrive/Dataset"
NEW_DATA_DIR = "/kaggle/input/arabic-food-101/Arabic Food 101 V3/Images"

# --- 2. RE-BUILD MASTER CLASS LIST (119 CLASSES) ---
# This is crucial to match the model's output
print("Re-building 119-class list...")
class_names_101_path = os.path.join(BASE_DIR_101, "meta/meta", "classes.txt")
with open(class_names_101_path, 'r') as f:
    original_classes = [line.strip() for line in f.readlines()]

new_class_folders = [d for d in os.listdir(NEW_DATA_DIR) if os.path.isdir(os.path.join(NEW_DATA_DIR, d))]
new_classes_cleaned = [name.lower().replace(" ", "_") for name in new_class_folders]

master_class_list = list(original_classes)
for new_class in new_classes_cleaned:
    if new_class not in master_class_list:
        master_class_list.append(new_class)

# We MUST sort it to match the training labels
master_class_list.sort()
NUM_TOTAL_CLASSES = len(master_class_list)

# Create a "pretty" version for printing
pretty_class_names = [name.replace("_", " ").title() for name in master_class_list]

print(f"Successfully loaded {NUM_TOTAL_CLASSES} total classes.")

# --- 3. DEFINE IMAGE TRANSFORM ---
# This MUST be the same as your validation transform
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
])

# --- 4. LOAD YOUR FINE-TUNED MODEL ---
print(f"Loading fine-tuned model from {FINAL_MODEL_PATH}")
# Load the base model structure with the NEW number of classes
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_TOTAL_CLASSES)

# Load your saved weights
model.load_state_dict(torch.load(FINAL_MODEL_PATH))
model.to(DEVICE)
model.eval() # Set model to evaluation mode
print("Model loaded successfully.")

# --- 5. PREDICTION FUNCTION ---
def predict_image(image_url):
    """
    Downloads, preprocesses, and predicts a single image from a URL.
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36"
    }

    try:
        response = requests.get(image_url, headers=headers)
        if response.status_code != 200:
            print(f"Error: Failed to download image. Status code: {response.status_code}")
            return

        img = Image.open(BytesIO(response.content)).convert("RGB")

    except Exception as e:
        print(f"Error loading image from URL: {e}")
        return

    # Preprocess the image
    img_tensor = test_transform(img)
    img_tensor = img_tensor.unsqueeze(0).to(DEVICE) # (C, H, W) -> (1, C, H, W)

    # Get predictions
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    # Get Top 5 predictions
    top5_prob, top5_idx = torch.topk(probabilities, 5)

    print("--- Top 5 Predictions ---")
    for i in range(top5_prob.size(0)):
        prob = top5_prob[i].item() * 100
        label = pretty_class_names[top5_idx[i].item()]
        print(f"{i+1}. {label:<25} ({prob:.2f}%)")

    # Display the image
    plt.imshow(img)
    plt.title(f"Prediction: {pretty_class_names[top5_idx[0].item()]}")
    plt.axis('off')
    plt.show()

# --- 6. TEST THE FUNCTION ---
# Try an image of Koshari!
image_url = "https://images.deliveryhero.io/image/talabat/MenuItems/C562E796505BD49C4817A83A70ED3BB7"
# Or Mahashi
# image_url = "https://www.chefindisguise.com/wp-content/uploads/2015/02/kousa-mahshi-FG.jpg"
# Or a Food-101 class
# image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a3/Eq_it-na_pizza-margherita_sep2005_sml.jpg/800px-Eq_it-na_pizza-margherita_sep2005_sml.jpg"

predict_image(image_url)

## Settings for Backend

In [ ]:
import os

# --- Paths for re-building the class list ---
BASE_DIR_101 = "/content/drive/MyDrive/Dataset"
NEW_DATA_DIR = "/kaggle/input/arabic-food-101/Arabic Food 101 V3/Images"

print("Re-building 119-class list...")
class_names_101_path = os.path.join(BASE_DIR_101, "meta/meta", "classes.txt")
with open(class_names_101_path, 'r') as f:
    original_classes = [line.strip() for line in f.readlines()]

new_class_folders = [d for d in os.listdir(NEW_DATA_DIR) if os.path.isdir(os.path.join(NEW_DATA_DIR, d))]
new_classes_cleaned = [name.lower().replace(" ", "_") for name in new_class_folders]

master_class_list = list(original_classes)
for new_class in new_classes_cleaned:
    if new_class not in master_class_list:
        master_class_list.append(new_class)

# We MUST sort it to match the training labels
master_class_list.sort()

# Create "pretty" names for the final output
pretty_class_names = [name.replace("_", " ").title() for name in master_class_list]

# Save the file
output_path = "food_119_classes.txt"
with open(output_path, 'w') as f:
    for name in pretty_class_names:
        f.write(f"{name}\n")

print(f"Successfully created '{output_path}' with {len(pretty_class_names)} classes.")
print("Please download this file from the Kaggle output directory.")

## Second Confusion Matrix

In [ ]:
pip install scikit-learn

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.amp import autocast
import timm
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
import numpy as np
import random
import warnings
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. CONFIGURATION ---
print("--- Configuration ---")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# --- Kaggle Paths ---
BASE_DIR_101 = "/content/drive/MyDrive/Dataset"
NEW_DATA_DIR = "/kaggle/input/arabic-food-101/Arabic Food 101 V3/Images"
# This is where your fine-tuning script saved the final model
FINAL_MODEL_PATH = "/kaggle/working/food_finetuned_model.pth"

# --- Model Config ---
MODEL_NAME = "vit_base_patch16_224.orig_in21k"
IMG_SIZE = 224

# --- Data Config ---
NUM_WORKERS = 2 # Safe number for Kaggle
PIN_MEMORY = False
BATCH_SIZE = 128 # Use a large batch size for fast eval

# --- 2. RE-BUILD MASTER CLASS LIST (119 CLASSES) ---
print("\n--- Re-building 119-class list ---")
class_names_101_path = os.path.join(BASE_DIR_101, "meta/meta", "classes.txt")
with open(class_names_101_path, 'r') as f:
    original_classes = [line.strip() for line in f.readlines()]

new_class_folders = [d for d in os.listdir(NEW_DATA_DIR) if os.path.isdir(os.path.join(NEW_DATA_DIR, d))]
new_classes_cleaned = [name.lower().replace(" ", "_") for name in new_class_folders]

master_class_list = list(original_classes)
for new_class in new_classes_cleaned:
    if new_class not in master_class_list:
        master_class_list.append(new_class)

master_class_list.sort()
NUM_TOTAL_CLASSES = len(master_class_list)
master_class_to_idx = {name: idx for idx, name in enumerate(master_class_list)}
print(f"Successfully loaded {NUM_TOTAL_CLASSES} total classes.")

# Create "pretty" names for the final report
pretty_class_names = {idx: name.replace("_", " ").title() for name, idx in master_class_to_idx.items()}

# --- 3. RE-BUILD THE COMBINED VALIDATION SET ---
print("\n--- Re-building the Combined Validation Set ---")
val_image_list = []

# Get original Food-101 test set
test_list_101_path = os.path.join(BASE_DIR_101, "meta/meta", "test.txt")
with open(test_list_101_path, 'r') as f:
    for line in f.readlines():
        line = line.strip()
        class_name = line.split('/')[0]
        img_path = os.path.join(BASE_DIR_101, "images", f"{line}.jpg")
        new_label = master_class_to_idx[class_name]
        val_image_list.append((img_path, new_label))
print(f"Loaded {len(val_image_list)} original validation images.")

# Get 20% validation split from Arabic Food V3
new_images_added_val = 0
for i, folder_name in enumerate(new_class_folders):
    class_name_cleaned = new_classes_cleaned[i]
    new_label = master_class_to_idx[class_name_cleaned]

    class_path = os.path.join(NEW_DATA_DIR, folder_name)
    images = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.seed(42) # Use a fixed seed so we get the *same* val set as in training
    random.shuffle(images)

    split_idx = int(len(images) * 0.8)

    for j, img_path in enumerate(images):
        if j >= split_idx: # Get the 20% validation split
            val_image_list.append((img_path, new_label))
            new_images_added_val += 1

print(f"Loaded {new_images_added_val} new validation images.")
print(f"TOTAL Validation Set Size: {len(val_image_list)} images.")

# --- 4. DATASET & DATALOADER ---
class FineTuneDataset(Dataset):
    def __init__(self, image_list, transform=None):
        self.image_list = image_list
        self.transform = transform
    def __len__(self):
        return len(self.image_list)
    def __getitem__(self, idx):
        img_path, label = self.image_list[idx]
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))
        if self.transform:
            img = self.transform(img)
        return img, label

# Use the same validation transforms as training
val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
])

val_dataset = FineTuneDataset(val_image_list, transform=val_transform)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

# --- 5. LOAD THE FINE-TUNED V1 MODEL ---
print(f"\n--- Loading fine-tuned model from {FINAL_MODEL_PATH} ---")

if not os.path.exists(FINAL_MODEL_PATH):
    print(f"ERROR: Model file not found at {FINAL_MODEL_PATH}")
    print("Please make sure your fine-tuning script (Cell 2) ran successfully and saved the file.")
else:
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_TOTAL_CLASSES)
    model.load_state_dict(torch.load(FINAL_MODEL_PATH, map_location=DEVICE))

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for evaluation.")
        model = nn.DataParallel(model)
    model.to(DEVICE)
    model.eval()
    print("Model loaded successfully.")

    # --- 6. RUN THE EVALUATION ---
    print("\n--- Running Evaluation ---")
    all_targets = []
    all_preds = []

    # Suppress warnings
    warnings.filterwarnings("ignore", message=".*`torch.cuda.amp.GradScaler(args...)` is deprecated.*")

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Evaluating"):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast('cuda', dtype=torch.float16):
                outputs = model(images)

            all_targets.extend(labels.cpu().numpy())
            all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())

    # --- 7. ANALYZE RESULTS & PRINT CONFUSION MATRIX ---
    print("\n--- Evaluation Complete: Analyzing Results ---")

    # 1. Generate the confusion matrix
    cm = confusion_matrix(all_targets, all_preds)

    # 2. Plot the full confusion matrix (as a heatmap)
    plt.figure(figsize=(20, 20))
    sns.heatmap(cm, annot=False, cbar=False, xticklabels=False, yticklabels=False)
    plt.title(f"Full Confusion Matrix ({NUM_TOTAL_CLASSES}x{NUM_TOTAL_CLASSES})", fontsize=20)
    plt.xlabel("Predicted", fontsize=16)
    plt.ylabel("True", fontsize=16)
    plt.show()

    # 3. Find and print the top 20 most confused pairs
    print("\n--- Top 20 Most Confused Pairs (V2 Plan) ---")

    # Set diagonal to 0 to ignore correct predictions
    np.fill_diagonal(cm, 0)

    # Get the indices of the top 20 flat array
    indices = np.argsort(cm.flatten())[-20:]
    top_pairs = []

    for idx in reversed(indices): # Iterate from largest to smallest
        true_idx, pred_idx = np.unravel_index(idx, cm.shape)
        count = cm[true_idx, pred_idx]
        if count == 0:
            continue

        true_label = pretty_class_names[true_idx]
        pred_label = pretty_class_names[pred_idx]
        top_pairs.append(f"Model confused '{true_label}' (True) with '{pred_label}' (Pred): {count} times")

    for pair in top_pairs:
        print(pair)

## Metric Learning

The V2 Strategy: Hard Augmentations + A "Tougher" Loss
Our Diagnosis: The Top 20 errors are all fine-grained pairs (Steak vs Filet Mignon). Our previous fine-tuning (Stage 2) was "soft." We used a basic CrossEntropyLoss and simple augmentations. This wasn't enough to force the model to learn the tiny, subtle differences.

In [ ]:
# --- 1. V2 CONFIGURATION ---
print("--- V2 Configuration ---")

# --- Paths ---
# We are loading the V1 fine-tuned model
V1_MODEL_PATH = "/kaggle/working/food_finetuned_model.pth"
# We will save our new V2 model here
V2_MODEL_SAVE_PATH = "/kaggle/working/food_V2_model.pth"

# --- V2 Training Config ---
V2_EPOCHS = 15   # We'll train for 15 epochs
V2_LR = 1e-6       # Start with the same tiny LR
V2_BATCH_SIZE = 64 # Keep the batch size small

In [ ]:
# --- 2. V2 DATA PREPARATION ---
print("--- V2 Data Preparation ---")

# (We need these variables from the previous V1 script)
# train_image_list, val_image_list, NUM_TOTAL_CLASSES

# --- NEW: V2 Training Transform with RandAugment ---
v2_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(),
    # Add our "hard" augmentation back in
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
])

# Validation transform (unchanged)
val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
])

# Create the V2 datasets
v2_train_dataset = FineTuneDataset(train_image_list, transform=v2_train_transform)
v2_val_dataset = FineTuneDataset(val_image_list, transform=val_transform)

# Create the V2 dataloaders
v2_train_loader = DataLoader(
    v2_train_dataset, batch_size=V2_BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    drop_last=True
)
v2_val_loader = DataLoader(
    v2_val_dataset, batch_size=V2_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

print(f"V2 DataLoaders created with {len(v2_train_dataset)} training images (with RandAugment).")

## Focal Loss Training

In [ ]:
# --- 3. V2 TRAINING (RandAugment + Mixup) ---
from timm.data import Mixup
import torch.nn.functional as F

# --- ADDING MISSING HELPER FUNCTIONS ---

def accuracy(output, target, topk=(1, 5)):
    """Computes the accuracy over the k top predictions for the specified values of k"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

def run_validation(model, loader, loss_fn):
    """Helper function to run a validation epoch."""
    model.eval()
    val_loss = 0.0
    val_acc1 = 0.0
    val_acc5 = 0.0

    val_loop = tqdm(loader, desc="           [Val]", leave=False)
    with torch.no_grad():
        for images, labels in val_loop:
            # Move data to GPU
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast('cuda', dtype=torch.float16):
                outputs = model(images)
                loss = loss_fn(outputs, labels)

            val_loss += loss.item()
            acc1, acc5 = accuracy(outputs, labels, topk=(1, 5))
            val_acc1 += acc1.item()
            val_acc5 += acc5.item()

    return val_loss / len(loader), val_acc1 / len(loader), val_acc5 / len(loader)

# --- END OF HELPER FUNCTIONS ---


print(f"\n--- Starting V2 Training ---")

# --- Load the V1 Model ---
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_TOTAL_CLASSES)
model.load_state_dict(torch.load(V1_MODEL_PATH, map_location=DEVICE))
print(f"Loaded V1 model from {V1_MODEL_PATH}")

# --- Unfreeze all layers ---
for param in model.parameters():
    param.requires_grad = True
print("Unfrozen all layers for V2 fine-tuning.")

model.to(DEVICE) # Move model to device *before* optimizer

# --- V2 Loss Function (Standard Cross Entropy) ---
loss_fn = nn.CrossEntropyLoss()
print("Using nn.CrossEntropyLoss().")

# --- Initialize Mixup ---
mixup_fn = Mixup(
    mixup_alpha=0.8,
    cutmix_alpha=1.0,
    prob=1.0,
    label_smoothing=0.1,
    num_classes=NUM_TOTAL_CLASSES
)
print("Initialized Mixup/CutMix.")

# --- V2 Optimizer & Scheduler ---
optimizer = optim.AdamW(model.parameters(), lr=V2_LR, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=V2_EPOCHS)
scaler = GradScaler()

# --- NEW: CHECKPOINT LOADING LOGIC ---
best_acc1 = 92.18 # Our starting point from V1
start_epoch = 0
checkpoint_path = "v2_checkpoint.pth"

if os.path.exists(checkpoint_path):
    print(f"V2 checkpoint found! Loading from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location='cpu')

    # We load the base model weights, not the DataParallel wrapper
    # This avoids the 'module.' prefix issue
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])

    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch']
    best_acc1 = checkpoint['best_acc1']

    print(f"Successfully loaded. Resuming training from epoch {start_epoch + 1}")

    # We must move the optimizer state to the GPU *after* loading
    print("Moving optimizer state to GPU...")
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(DEVICE)
else:
    print("No V2 checkpoint found. Starting V2 training from scratch.")
# --- END OF CHECKPOINT LOGIC ---


# --- Apply DataParallel AFTER loading and moving model ---
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for V2 training.")
    model = nn.DataParallel(model)

# --- V2 Training Loop ---
# Loop now starts from 'start_epoch'
for epoch in range(start_epoch, V2_EPOCHS):
    model.train()
    train_loss = 0.0

    train_loop = tqdm(v2_train_loader, desc=f"Epoch {epoch+1}/{V2_EPOCHS} [V2 Train]", leave=False)

    for images, labels in train_loop:
        # Apply Mixup on the CPU
        images, labels_mixed = mixup_fn(images, labels)

        # Move data to GPU
        images = images.to(DEVICE, non_blocking=True)
        labels_mixed = labels_mixed.to(DEVICE, non_blocking=True)

        with autocast('cuda', dtype=torch.float16):
            outputs = model(images)
            loss = loss_fn(outputs, labels_mixed)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item(), lr=optimizer.param_groups[0]['lr'])

    avg_train_loss = train_loss / len(v2_train_loader)

    # Run validation
    val_loss_fn = nn.CrossEntropyLoss()
    avg_val_loss, avg_val_acc1, avg_val_acc5 = run_validation(model, v2_val_loader, val_loss_fn)

    scheduler.step()

    print(f"Epoch {epoch+1}/{V2_EPOCHS} [V2] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc@1: {avg_val_acc1:.2f}%")

    # Save the best model
    if avg_val_acc1 > best_acc1:
        best_acc1 = avg_val_acc1
        print(f"🎉 New V2 best model! Saving to '{V2_MODEL_SAVE_PATH}' with Acc@1: {best_acc1:.2f}%")

        model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(model_state, V2_MODEL_SAVE_PATH)

    # Save checkpoint
    model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    checkpoint = {
        'epoch': epoch + 1, # Save the *next* epoch number
        'model_state_dict': model_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_acc1': best_acc1,
    }
    torch.save(checkpoint, checkpoint_path)

print(f"\n--- V2 Training Finished ---")
print(f"Best V2 Validation Acc@1: {best_acc1:.2f}%")

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.amp import autocast
import timm
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
import numpy as np
import random
import warnings
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. CONFIGURATION ---
print("--- V2 Model Evaluation ---")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# --- Kaggle Paths ---
BASE_DIR_101 = "/content/drive/MyDrive/Dataset"
NEW_DATA_DIR = "/kaggle/input/arabic-food-101/Arabic Food 101 V3/Images"
# --- This is our new V2 model ---
FINAL_MODEL_PATH = "/kaggle/working/food_V2_model.pth"

# --- Model Config ---
MODEL_NAME = "vit_base_patch16_224.orig_in21k"
IMG_SIZE = 224

# --- Data Config ---
# --- Using num_workers=0 is the safest way to prevent "Aw, Snap!" crashes ---
NUM_WORKERS = 0
PIN_MEMORY = False
BATCH_SIZE = 128 # Use a large batch size for fast eval

# --- 2. RE-BUILD MASTER CLASS LIST (119 CLASSES) ---
print("\n--- Re-building 119-class list ---")
class_names_101_path = os.path.join(BASE_DIR_101, "meta/meta", "classes.txt")
with open(class_names_101_path, 'r') as f:
    original_classes = [line.strip() for line in f.readlines()]

new_class_folders = [d for d in os.listdir(NEW_DATA_DIR) if os.path.isdir(os.path.join(NEW_DATA_DIR, d))]
new_classes_cleaned = [name.lower().replace(" ", "_") for name in new_class_folders]

master_class_list = list(original_classes)
for new_class in new_classes_cleaned:
    if new_class not in master_class_list:
        master_class_list.append(new_class)

master_class_list.sort()
NUM_TOTAL_CLASSES = len(master_class_list)
master_class_to_idx = {name: idx for idx, name in enumerate(master_class_list)}
print(f"Successfully loaded {NUM_TOTAL_CLASSES} total classes.")

# Create "pretty" names for the final report
pretty_class_names = {idx: name.replace("_", " ").title() for name, idx in master_class_to_idx.items()}

# --- 3. RE-BUILD THE COMBINED VALIDATION SET ---
print("\n--- Re-building the Combined Validation Set ---")

# *** THIS IS THE FIX ***
val_image_list = [] # Was val_image_.list = []

# Get original Food-101 test set
test_list_101_path = os.path.join(BASE_DIR_101, "meta/meta", "test.txt")
with open(test_list_101_path, 'r') as f:
    for line in f.readlines():
        line = line.strip()
        class_name = line.split('/')[0]
        img_path = os.path.join(BASE_DIR_101, "images", f"{line}.jpg")
        new_label = master_class_to_idx[class_name]
        val_image_list.append((img_path, new_label))
print(f"Loaded {len(val_image_list)} original validation images.")

# Get 20% validation split from Arabic Food V3
new_images_added_val = 0
for i, folder_name in enumerate(new_class_folders):
    class_name_cleaned = new_classes_cleaned[i]
    new_label = master_class_to_idx[class_name_cleaned]

    class_path = os.path.join(NEW_DATA_DIR, folder_name)
    images = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.seed(42) # Use a fixed seed so we get the *same* val set as in training
    random.shuffle(images)

    split_idx = int(len(images) * 0.8)

    for j, img_path in enumerate(images):
        if j >= split_idx: # Get the 20% validation split
            val_image_list.append((img_path, new_label))
            new_images_added_val += 1

print(f"Loaded {new_images_added_val} new validation images.")
print(f"TOTAL Validation Set Size: {len(val_image_list)} images.")

# --- 4. DATASET & DATALOADER ---
class FineTuneDataset(Dataset):
    def __init__(self, image_list, transform=None):
        self.image_list = image_list
        self.transform = transform
    def __len__(self):
        return len(self.image_list)
    def __getitem__(self, idx):
        img_path, label = self.image_list[idx]
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))
        if self.transform:
            img = self.transform(img)
        return img, label

# Use the same validation transforms as training
val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
])

val_dataset = FineTuneDataset(val_image_list, transform=val_transform)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

# --- 5. LOAD THE V2 MODEL ---
print(f"\n--- Loading V2 model from {FINAL_MODEL_PATH} ---")

if not os.path.exists(FINAL_MODEL_PATH):
    print(f"ERROR: Model file not found at {FINAL_MODEL_PATH}")
else:
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_TOTAL_CLASSES)
    model.load_state_dict(torch.load(FINAL_MODEL_PATH, map_location=DEVICE))

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for evaluation.")
        model = nn.DataParallel(model)
    model.to(DEVICE)
    model.eval()
    print("V2 Model loaded successfully.")

    # --- 6. RUN THE EVALUATION ---
    print("\n--- Running Evaluation on V2 Model ---")
    all_targets = []
    all_preds = []

    # Suppress warnings
    warnings.filterwarnings("ignore")

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Evaluating V2"):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast('cuda', dtype=torch.float16):
                outputs = model(images)

            all_targets.extend(labels.cpu().numpy())
            all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())

    # --- 7. ANALYZE RESULTS & PRINT CONFUSION MATRIX ---
    print("\n--- Evaluation Complete: V2 Analysis ---")

    # 1. Calculate and print final accuracy
    final_accuracy = accuracy_score(all_targets, all_preds)
    print(f"Final V2 Model Accuracy: {final_accuracy * 100:.2f}%")

    # 2. Generate the confusion matrix
    cm = confusion_matrix(all_targets, all_preds)

    # 3. Plot the full confusion matrix (as a heatmap)
    plt.figure(figsize=(20, 20))
    sns.heatmap(cm, annot=False, cbar=False, xticklabels=False, yticklabels=False)
    plt.title(f"V2 Model Confusion Matrix ({NUM_TOTAL_CLASSES}x{NUM_TOTAL_CLASSES})", fontsize=20)
    plt.xlabel("Predicted", fontsize=16)
    plt.ylabel("True", fontsize=16)
    plt.show()

    # 4. Find and print the top 20 most confused pairs
    print("\n--- V2 Model: Top 20 Most Confused Pairs ---")

    np.fill_diagonal(cm, 0)
    indices = np.argsort(cm.flatten())[-20:]
    top_pairs = []

    for idx in reversed(indices):
        true_idx, pred_idx = np.unravel_index(idx, cm.shape)
        count = cm[true_idx, pred_idx]
        if count == 0:
            continue

        true_label = pretty_class_names[true_idx]
        pred_label = pretty_class_names[pred_idx]
        top_pairs.append(f"Model confused '{true_label}' (True) with '{pred_label}' (Pred): {count} times")

    for pair in top_pairs:
        print(pair)

## Test from the internet

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
import timm
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from PIL import Image
import requests
from io import BytesIO
import os
import matplotlib.pyplot as plt

# --- 1. CONFIGURATION ---
print("--- Setting up V2 Internet Test ---")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# --- Paths (Kaggle Environment) ---
BASE_DIR_101 = "/content/drive/MyDrive/Dataset"
NEW_DATA_DIR = "/kaggle/input/arabic-food-101/Arabic Food 101 V3/Images"
MODEL_PATH = "/kaggle/working/food_V2_model.pth"

# --- Model Config ---
# We must match the architecture used during training
MODEL_NAME = "vit_base_patch16_224.orig_in21k"
IMG_SIZE = 224

# --- 2. RE-BUILD CLASS LIST (119 CLASSES) ---
# We need to reconstruct the exact list of classes the model was trained on.
print("Re-building class list...")

# A. Load 101 Original Classes
class_names_101_path = os.path.join(BASE_DIR_101, "meta/meta", "classes.txt")
with open(class_names_101_path, 'r') as f:
    original_classes = [line.strip() for line in f.readlines()]

# B. Load New Arabic/Egyptian Classes
new_class_folders = [d for d in os.listdir(NEW_DATA_DIR) if os.path.isdir(os.path.join(NEW_DATA_DIR, d))]
new_classes_cleaned = [name.lower().replace(" ", "_") for name in new_class_folders]

# C. Merge and Sort
master_class_list = list(original_classes)
for new_class in new_classes_cleaned:
    if new_class not in master_class_list:
        master_class_list.append(new_class)

master_class_list.sort() # Critical: Must be sorted alphabetically
NUM_CLASSES = len(master_class_list)

# Create a "pretty" version for display (e.g., "fava_beans" -> "Fava Beans")
pretty_class_names = [name.replace("_", " ").title() for name in master_class_list]
print(f"Successfully loaded {NUM_CLASSES} classes.")

# --- 3. PREPARE THE MODEL ---
print(f"Loading V2 Model from {MODEL_PATH}...")

if not os.path.exists(MODEL_PATH):
    print(f"ERROR: Model file not found at {MODEL_PATH}")
    print("Please check that you have run the V2 training script successfully.")
else:
    # Create the model architecture
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)

    # Load the weights
    # map_location ensures it loads even if you switched devices
    state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)

    model.to(DEVICE)
    model.eval() # Set to evaluation mode (freezes Dropout/BatchNorm)
    print("V2 Model loaded and ready!")

# --- 4. DEFINE TRANSFORMS ---
# Must match validation transforms exactly
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
])

# --- 5. PREDICTION FUNCTION ---
def predict_url(url):
    """
    Downloads an image from a URL, pre-processes it, and prints the Top 3 predictions.
    """
    print(f"\n--- Testing URL: {url} ---")

    # A. Download
    headers = {'User-Agent': 'Mozilla/5.0'} # Pretend to be a browser
    try:
        response = requests.get(url, headers=headers, timeout=5)
        if response.status_code != 200:
            print(f"Error: Failed to download (Status {response.status_code})")
            return
        img = Image.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        print(f"Error: Could not process image. {e}")
        return

    # B. Pre-process
    img_tensor = test_transform(img)
    img_tensor = img_tensor.unsqueeze(0).to(DEVICE) # Add batch dimension

    # C. Predict
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    # D. Results
    top_prob, top_idx = torch.topk(probabilities, 3) # Get top 3

    # Display image
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Top Pred: {pretty_class_names[top_idx[0]]} ({top_prob[0]*100:.1f}%)")
    plt.show()

    # Print text results
    for i in range(3):
        idx = top_idx[i].item()
        score = top_prob[i].item() * 100
        name = pretty_class_names[idx]
        print(f"{i+1}. {name:<20} ... {score:.2f}%")

# --- 6. RUN TESTS ---

# Test 1: Koshari (New Class)
predict_url("https://www.simplyleb.com/wp-content/uploads/2021/05/Koshari.jpg")

# Test 2: Falafel (Merged Class)
predict_url("https://upload.wikimedia.org/wikipedia/commons/thumb/2/20/Falafel_balls_in_a_pita.jpg/800px-Falafel_balls_in_a_pita.jpg")

# Test 3: Hamburger (Original Class)
predict_url("https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/Hamburger_%28black_bg%29.jpg/800px-Hamburger_%28black_bg%29.jpg")

# Test 4: Mahashi (New Class)
predict_url("https://upload.wikimedia.org/wikipedia/commons/thumb/7/7f/Stuffed_Squash.jpg/800px-Stuffed_Squash.jpg")

# V3 Model

What's New in V3?
Architecture: ConvNeXt Base (convnext_base.fb_in22k)

This is a CNN that "thinks" like a Transformer. It uses large kernels (7x7) and modern layer designs. It is often more robust on smaller, messy datasets than ViT.

We use the version pre-trained on ImageNet-22k (massive dataset) for maximum knowledge transfer.

Optimizer: SAM (Sharpness-Aware Minimization)

Standard SGD/Adam seeks the lowest loss.

SAM seeks the flattest loss landscape.

Why it matters: A "flat" minimum means that if the test data is slightly different (e.g., different lighting in your Egyptian food photos), the error won't spike. It is the gold standard for generalization right now.

Implementation Detail:

SAM requires two forward passes per batch. It effectively doubles the training time per epoch, but it usually converges to a higher accuracy.

In [ ]:
pip install timm

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler
from torch.amp import autocast
import timm
from timm.data import Mixup
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from PIL import Image, UnidentifiedImageError
import numpy as np
import random
import warnings
import time

# --- 1. CONFIGURATION ---
print("--- V3 Configuration: ConvNeXt Base + AdamW (Stable) ---")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Paths
BASE_DIR_101 = "/content/drive/MyDrive/Dataset"
NEW_DATA_DIR = "/kaggle/input/arabic-food-101/Arabic Food 101 V3/Images"
V3_MODEL_SAVE_PATH = "/kaggle/working/food_V3_convnext.pth"
CHECKPOINT_PATH = "v3_checkpoint.pth"

# Model Config
MODEL_NAME = "convnext_base.fb_in22k"
IMG_SIZE = 224

# Training Config
EPOCHS = 25
BATCH_SIZE = 32 # ConvNeXt with AdamW uses less memory than SAM, 32 might work. If crash, use 16.
NUM_WORKERS = 2 # Keep 0 to prevent "Aw Snap"
LR = 1e-4       # Standard stable LR for ConvNeXt

# --- 2. DATA PREPARATION ---
print("Preparing Data...")
class_names_101_path = os.path.join(BASE_DIR_101, "meta/meta", "classes.txt")
with open(class_names_101_path, 'r') as f:
    original_classes = [line.strip() for line in f.readlines()]

new_class_folders = [d for d in os.listdir(NEW_DATA_DIR) if os.path.isdir(os.path.join(NEW_DATA_DIR, d))]
new_classes_cleaned = [name.lower().replace(" ", "_") for name in new_class_folders]

master_class_list = list(original_classes)
for new_class in new_classes_cleaned:
    if new_class not in master_class_list:
        master_class_list.append(new_class)
master_class_list.sort()
NUM_TOTAL_CLASSES = len(master_class_list)
master_class_to_idx = {name: idx for idx, name in enumerate(master_class_list)}

train_image_list = []
val_image_list = []

# Food-101 Data
train_list_101_path = os.path.join(BASE_DIR_101, "meta/meta", "train.txt")
with open(train_list_101_path, 'r') as f:
    for line in f.readlines():
        line = line.strip()
        class_name = line.split('/')[0]
        img_path = os.path.join(BASE_DIR_101, "images", f"{line}.jpg")
        train_image_list.append((img_path, master_class_to_idx[class_name]))

test_list_101_path = os.path.join(BASE_DIR_101, "meta/meta", "test.txt")
with open(test_list_101_path, 'r') as f:
    for line in f.readlines():
        line = line.strip()
        class_name = line.split('/')[0]
        img_path = os.path.join(BASE_DIR_101, "images", f"{line}.jpg")
        val_image_list.append((img_path, master_class_to_idx[class_name]))

# Arabic Data
for i, folder_name in enumerate(new_class_folders):
    class_name_cleaned = new_classes_cleaned[i]
    new_label = master_class_to_idx[class_name_cleaned]
    class_path = os.path.join(NEW_DATA_DIR, folder_name)
    images = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.seed(42); random.shuffle(images)
    split_idx = int(len(images) * 0.8)
    for j, img_path in enumerate(images):
        if j < split_idx: train_image_list.append((img_path, new_label))
        else: val_image_list.append((img_path, new_label))

class FineTuneDataset(Dataset):
    def __init__(self, image_list, transform=None):
        self.image_list = image_list
        self.transform = transform
    def __len__(self): return len(self.image_list)
    def __getitem__(self, idx):
        img_path, label = self.image_list[idx]
        try: img = Image.open(img_path).convert("RGB")
        except: img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))
        if self.transform: img = self.transform(img)
        return img, label

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
])
val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
])

train_dataset = FineTuneDataset(train_image_list, transform=train_transform)
val_dataset = FineTuneDataset(val_image_list, transform=val_transform)

print("Creating DataLoaders...")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=False, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

# --- 4. MODEL SETUP ---
print(f"--- Loading {MODEL_NAME} ---")
# Create ConvNeXt
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_TOTAL_CLASSES)
model.to(DEVICE)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

# --- 5. OPTIMIZER & LOSS (STANDARD STABLE SETUP) ---
mixup_fn = Mixup(mixup_alpha=0.8, cutmix_alpha=1.0, prob=1.0, label_smoothing=0.1, num_classes=NUM_TOTAL_CLASSES)
loss_fn = nn.CrossEntropyLoss()

# Using standard AdamW - No SAM, No tricks.
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler() # Standard Scaler

# --- 6. CHECKPOINT LOADING ---
start_epoch = 0
best_acc1 = 0.0

if os.path.exists(CHECKPOINT_PATH):
    print(f"Checkpoint found! Loading from {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')

    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])

    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch']
    best_acc1 = checkpoint['best_acc1']

    print(f"Successfully loaded. Resuming from Epoch {start_epoch + 1}")

    print("Moving optimizer state to GPU...")
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(DEVICE)
else:
    print("No checkpoint found. Starting fresh.")

# --- 7. HELPERS ---
def accuracy(output, target, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

def run_validation(model, loader):
    model.eval()
    val_acc1, val_acc5 = 0.0, 0.0
    print("Running Validation...", end="")
    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            with autocast('cuda', dtype=torch.float16):
                outputs = model(images)
            acc1, acc5 = accuracy(outputs, labels, topk=(1, 5))
            val_acc1 += acc1.item()
            val_acc5 += acc5.item()
            if i % 100 == 0: print(".", end="")
    print(" Done.")
    return val_acc1 / len(loader), val_acc5 / len(loader)

# --- 8. TRAINING LOOP ---
print(f"--- Starting V3 Training from Epoch {start_epoch+1} ---")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_loss = 0.0
    start_time = time.time()

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        images, labels_mixed = mixup_fn(images, labels)

        # Standard Forward Pass
        with autocast('cuda', dtype=torch.float16):
            outputs = model(images)
            loss = loss_fn(outputs, labels_mixed)

        # Standard Backward Pass (Stable)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        train_loss += loss.item()

        if i % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    scheduler.step()

    # Validate
    val_acc1, val_acc5 = run_validation(model, val_loader)
    epoch_duration = time.time() - start_time
    print(f"Epoch {epoch+1} Finished | Time: {epoch_duration:.0f}s | Train Loss: {train_loss/len(train_loader):.4f} | Val Acc@1: {val_acc1:.2f}%")

    # Save Best
    if val_acc1 > best_acc1:
        best_acc1 = val_acc1
        print(f"🎉 New Best V3 Model! Acc: {best_acc1:.2f}%")
        save_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(save_dict, V3_MODEL_SAVE_PATH)

    # Save Checkpoint
    model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_acc1': best_acc1,
    }
    torch.save(checkpoint, CHECKPOINT_PATH)
    print("Checkpoint saved.")

print(f"--- V3 Training Finished. Best Acc: {best_acc1:.2f}% ---")

## Evaluation Script

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.amp import autocast
import timm
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
import numpy as np
import random
import warnings
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. CONFIGURATION ---
print("--- V3 Model Evaluation (ConvNeXt) ---")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Paths
BASE_DIR_101 = "/content/drive/MyDrive/Dataset"
NEW_DATA_DIR = "/kaggle/input/arabic-food-101/Arabic Food 101 V3/Images"
# Loading the V3 model
FINAL_MODEL_PATH = "/kaggle/working/food_V3_convnext.pth"

# Model Config (Must match training!)
MODEL_NAME = "convnext_base.fb_in22k"
IMG_SIZE = 224

# Data Config
NUM_WORKERS = 0
PIN_MEMORY = False
BATCH_SIZE = 32 # Eval uses less memory than training

# --- 2. RE-BUILD CLASS LIST ---
print("\n--- Re-building 119-class list ---")
class_names_101_path = os.path.join(BASE_DIR_101, "meta/meta", "classes.txt")
with open(class_names_101_path, 'r') as f:
    original_classes = [line.strip() for line in f.readlines()]

new_class_folders = [d for d in os.listdir(NEW_DATA_DIR) if os.path.isdir(os.path.join(NEW_DATA_DIR, d))]
new_classes_cleaned = [name.lower().replace(" ", "_") for name in new_class_folders]

master_class_list = list(original_classes)
for new_class in new_classes_cleaned:
    if new_class not in master_class_list:
        master_class_list.append(new_class)

master_class_list.sort()
NUM_TOTAL_CLASSES = len(master_class_list)
master_class_to_idx = {name: idx for idx, name in enumerate(master_class_list)}
pretty_class_names = {idx: name.replace("_", " ").title() for name, idx in master_class_to_idx.items()}
print(f"Loaded {NUM_TOTAL_CLASSES} classes.")

# --- 3. RE-BUILD VALIDATION SET ---
print("\n--- Re-building Validation Set ---")
val_image_list = []

# Food-101 Test
test_list_101_path = os.path.join(BASE_DIR_101, "meta/meta", "test.txt")
with open(test_list_101_path, 'r') as f:
    for line in f.readlines():
        line = line.strip()
        class_name = line.split('/')[0]
        img_path = os.path.join(BASE_DIR_101, "images", f"{line}.jpg")
        val_image_list.append((img_path, master_class_to_idx[class_name]))

# Arabic V3 Test (20% split)
for i, folder_name in enumerate(new_class_folders):
    class_name_cleaned = new_classes_cleaned[i]
    new_label = master_class_to_idx[class_name_cleaned]
    class_path = os.path.join(NEW_DATA_DIR, folder_name)
    images = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.seed(42); random.shuffle(images)
    split_idx = int(len(images) * 0.8)
    for j, img_path in enumerate(images):
        if j >= split_idx:
            val_image_list.append((img_path, new_label))

print(f"Validation Set Size: {len(val_image_list)} images.")

# --- 4. DATASET ---
class FineTuneDataset(Dataset):
    def __init__(self, image_list, transform=None):
        self.image_list = image_list
        self.transform = transform
    def __len__(self): return len(self.image_list)
    def __getitem__(self, idx):
        img_path, label = self.image_list[idx]
        try: img = Image.open(img_path).convert("RGB")
        except: img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))
        if self.transform: img = self.transform(img)
        return img, label

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
])

val_dataset = FineTuneDataset(val_image_list, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# --- 5. LOAD V3 MODEL ---
print(f"\n--- Loading V3 Model: {MODEL_NAME} ---")
if not os.path.exists(FINAL_MODEL_PATH):
    print(f"ERROR: {FINAL_MODEL_PATH} not found!")
else:
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_TOTAL_CLASSES)
    # Load weights
    checkpoint = torch.load(FINAL_MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint)

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(DEVICE)
    model.eval()
    print("V3 Model loaded successfully.")

    # --- 6. EVALUATION ---
    print("\n--- Running Evaluation ---")
    all_targets = []
    all_preds = []
    warnings.filterwarnings("ignore")

    with torch.no_grad():
        # Using tqdm here is usually safe for eval as it's faster than training
        for images, labels in tqdm(val_loader, desc="Evaluating V3"):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast('cuda', dtype=torch.float16):
                outputs = model(images)

            all_targets.extend(labels.cpu().numpy())
            all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())

    # --- 7. RESULTS ---
    acc = accuracy_score(all_targets, all_preds)
    print(f"\nFinal V3 Accuracy: {acc * 100:.2f}%")

    cm = confusion_matrix(all_targets, all_preds)

    # Plot Heatmap
    plt.figure(figsize=(20, 20))
    sns.heatmap(cm, annot=False, cbar=False, xticklabels=False, yticklabels=False)
    plt.title(f"V3 Confusion Matrix ({NUM_TOTAL_CLASSES} classes)", fontsize=20)
    plt.show()

    # Top 20 Errors
    print("\n--- V3 Model: Top 20 Most Confused Pairs ---")
    np.fill_diagonal(cm, 0)
    indices = np.argsort(cm.flatten())[-20:]
    top_pairs = []

    for idx in reversed(indices):
        true_idx, pred_idx = np.unravel_index(idx, cm.shape)
        count = cm[true_idx, pred_idx]
        if count == 0: continue
        true_label = pretty_class_names[true_idx]
        pred_label = pretty_class_names[pred_idx]
        top_pairs.append(f"Model confused '{true_label}' with '{pred_label}': {count} times")

    for pair in top_pairs:
        print(pair)

# PART — Link to Nutrisense-AI  (photo → food → disease risk)

This classifier names the food in a photo. **Nutrisense-AI** turns that food into nutrients and
predicts the diseases the person is most at risk of — anemia, diabetes, overweight. Here we wire the
two together using the same conventions as the Nutrisense full-project notebook (FCD, `FCD_TO_MODEL`, `MEALS_PER_DAY`, the `nutrisense_model.joblib` bundle).

```
  photo → recognize() [this notebook] → food label
        → Rwandan FCD → nutrients → scale to a day → risk model [Nutrisense] → risk
```

**Honest scope.** Food-101 (+Arabic) are Western/Middle-Eastern dishes, so this CNN is the *methodology
demo*; only the foods present in `FOOD101_TO_FCD` map to nutrients, the rest fall back to a typed meal.
The deployed Rwandan recogniser (Nutrisense Part 5A) handles local foods. Both feed the **same** risk model.


### L1 — Wrap your trained model as `recognize(image)`
Point `MODEL_PATH` at whichever checkpoint you trust most (V1 `food101_best_model.pth`, the fine-tuned
`food_finetuned_model.pth`, or `food_V2_model.pth`). Same transform you validated with.


In [ ]:
import torch, timm, os
from torchvision import transforms
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from PIL import Image

MODEL_NAME   = "vit_base_patch16_224.orig_in21k"
BASE_DIR     = "/content/drive/MyDrive/Dataset"
MODEL_PATH   = os.path.join(BASE_DIR, "food101_best_model.pth")   # FIX: load from Drive
CLASSES_PATH = os.path.join(BASE_DIR, "meta/meta/classes.txt")    # FIX: Colab/Drive path
IMG_SIZE = 224
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

with open(CLASSES_PATH) as f:
    CLASS_NAMES = [l.strip() for l in f]            # e.g. 'fried_rice'  (use your 119-class list if fine-tuned)

_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE+32, IMG_SIZE+32)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD),
])
_model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=len(CLASS_NAMES))
_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
_model.to(DEVICE).eval()

def recognize(image_path):
    """photo -> (food_label, confidence)"""
    img = Image.open(image_path).convert("RGB")
    x = _tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p = torch.softmax(_model(x), dim=1)[0]
    i = int(p.argmax())
    return CLASS_NAMES[i], float(p[i])
print("recognizer ready")


### L2 — Food label → nutrients (Rwandan FCD, Nutrisense convention)
Same nutrient keys Nutrisense uses: `energy, protein, iron, vitA, vitC, fiber, calcium, zinc`.


In [ ]:
import pandas as pd, numpy as np
NUTRIENTS = ['energy','protein','iron','vitA','vitC','fiber','calcium','zinc']
# per 100 g (matches the Nutrisense FCD)
FCD = {
 'ubugali':   dict(name='Ubugali (cassava/maize)', vals=[110,1.5,0.5,0, 0, 1.0,8,  0.5]),
 'isombe':    dict(name='Isombe (cassava leaves)', vals=[50, 5.0,3.5,350,60,3.0,120,0.8]),
 'ibishyimbo':dict(name='Ibishyimbo (beans)',      vals=[130,8.0,2.5,0, 1, 6.0,50, 1.2]),
 'umuceri':   dict(name='Umuceri (rice)',          vals=[130,2.7,0.4,0, 0, 0.4,10, 0.5]),
 'imboga':    dict(name='Imboga (greens)',         vals=[30, 3.0,2.3,300,40,2.0,200,0.5]),
 'inyama':    dict(name='Inyama (beef)',           vals=[215,26, 2.6,0, 0, 0,  12, 4.8]),
 'ibirayi':   dict(name='Ibirayi (potato)',        vals=[87, 2.0,0.4,0, 13,1.8,12, 0.3]),
 'avoka':     dict(name='Avoka (avocado)',         vals=[160,2.0,0.6,7, 10,7.0,12, 0.6]),
 'igitoki':   dict(name='Igitoki (plantain)',      vals=[122,1.3,0.6,30,11,2.3,3,  0.2]),
}
# Food-101 / Arabic class -> closest Rwandan FCD food. Extend as you collect Rwandan images.
FOOD101_TO_FCD = {
 'fried_rice':'umuceri','risotto':'umuceri','french_fries':'ibirayi','baked_potato':'ibirayi',
 'steak':'inyama','filet_mignon':'inyama','prime_rib':'inyama','hamburger':'inyama','pork_chop':'inyama',
 'edamame':'ibishyimbo','falafel':'ibishyimbo','hummus':'ibishyimbo','guacamole':'avoka',
}
def label_to_meal(label, grams=300):
    key = FOOD101_TO_FCD.get(label.lower().replace(' ','_'))
    return [(key, grams)] if key else None
def meal_to_nutrients(meal):
    tot = {n:0.0 for n in NUTRIENTS}
    for k,g in meal:
        for n,v in zip(NUTRIENTS, FCD[k]['vals']): tot[n] += v*g/100
    return tot


### L3 — Load the Nutrisense risk model and predict
Upload `nutrisense_model.joblib` (saved to `MyDrive/Dataset/processed` by the Nutrisense notebook)
as a Kaggle dataset, and point `BUNDLE_PATH` at it. Same scaling logic as Nutrisense Part 6.


In [ ]:
import joblib
BUNDLE_PATH = "/content/drive/MyDrive/Dataset/processed/nutrisense_model.joblib"  # FIX: Drive path from the Nutrisense notebook
bundle = joblib.load(BUNDLE_PATH)        # bundle['features'], bundle['models'] (label -> sklearn pipeline)

FCD_TO_MODEL = {'energy':'food_energy_kcal','protein':'food_protein_g','iron':'food_iron_mg',
                'vitC':'food_vitC_mg','vitA':'food_vitA_rae_mcg','fiber':'food_fiber_g',
                'calcium':'food_calcium_mg','zinc':'food_zinc_mg'}
MEALS_PER_DAY = 3   # one photo = one meal; scale to a day (model trained on daily totals)

def build_features(profile, meal):
    sex = profile.get('sex',2)
    sex = (1 if str(sex).lower().startswith('m') else 2) if isinstance(sex,str) else sex
    feats = {'age':float(profile.get('age',0)),'sex':float(sex),'pregnant':float(profile.get('pregnant',0))}
    for n,v in meal_to_nutrients(meal).items():
        if n in FCD_TO_MODEL: feats[FCD_TO_MODEL[n]] = v*MEALS_PER_DAY
    return feats

def predict_risk(feats):
    x = pd.DataFrame([feats]).reindex(columns=bundle['features']).fillna(0)
    return {t.replace('label_',''): round(float(m.predict_proba(x)[:,1][0]),3)
            for t,m in bundle['models'].items()}


### L4 — End to end: a photo in, a disease risk out


In [ ]:
def photo_to_risk(image_path, profile):
    label, conf = recognize(image_path)
    print(f'recognised: {label}  ({conf:.0%})')
    meal = label_to_meal(label)
    if meal is None:
        print(f"  '{label}' not in FOOD101_TO_FCD yet -> using a typed Rwandan meal")
        meal = [('ubugali',300),('ibishyimbo',120),('imboga',100)]
    risk = predict_risk(build_features(profile, meal))
    print('\npredicted disease risk:')
    for d,p in sorted(risk.items(), key=lambda x:-x[1]):
        band = 'lower' if p<0.34 else 'moderate' if p<0.6 else 'HIGHER'
        print(f'  {d:11s} {p*100:4.0f}%  ({band})')
    return risk

# Example (uncomment with a real meal image path):
# photo_to_risk('/content/drive/MyDrive/Dataset/images/steak/100135.jpg', {'age':28,'sex':'female'})


### L5 — Notes for your thesis
- **The recogniser is the input; the risk model is the contribution.** This notebook proves the CNN
  methodology; the disease prediction comes from the Nutrisense model trained on NHANES.
- **Western foods are a placeholder.** For real use, fine-tune this classifier on **Rwandan food
  images** you collect, then replace `FOOD101_TO_FCD` with a direct class→FCD map. Steps L2–L4 don't change.
- **One photo is one meal**, scaled to a day (`MEALS_PER_DAY`); accumulating several photos over days
  gives a truer profile, as in the Nutrisense pipeline.
- **Keep manual food entry as a fallback** so the app still works when a dish is misrecognised or
  isn't in the model yet.
